In [ ]:
#| default_exp read

## Reading and inspection

Readable notebook views plus implementation-oriented context for co-creation.

The preferred public context API is `context(target, scope=".")`. It covers repository orientation, one-file inspection, one chapter, one cell id, and one concrete implementation without exposing raw notebook JSON.

Notebook chapters use a code-first rhythm: exported code, trailing Markdown explanation, example cells, focused test cells, then the next exported code. The readers preserve that source order and associate a symbol with the cells that follow it.

The focused readers remain small building blocks for the unified function.

Reading is the first problem to solve for notebook automation. An agent should not have to inspect raw `.ipynb` JSON just to learn what a project, notebook, chapter, or implementation contains, and a human reviewer should not have to scroll through outputs and metadata to find the code.

This notebook builds compact context views for four common questions: what project is this, what is in this file, what belongs to this chapter, and what should I know before changing this symbol.

For a chapter, read the story in this order:

```
## Chapter
- exported code
- trailing explanation
- example(s)
- test(s)
- exported code
- trailing explanation
- example(s)
- test(s)
- ...
```

Symbol readers use the same rhythm: implementation first, then trailing Docs, then the following example and test cells. They stop the local group at the next exported code cell so one symbol's explanation does not leak into the next one.

The reader is intentionally a triage tool before it is a renderer. Start with `context(target, scope=".")`; use `target="project"`, a notebook path/name, a chapter title, a cell id, or a Python symbol.

```python
context("project", scope="nbs")
context("nbs/02_write.ipynb")
context("write_nb", scope="nbs/02_write.ipynb")
```

## Reader APIs

Start with the unified `context` reader for triage. The focused readers remain useful when a caller needs one stable output shape.

#### Production contract

The reading tools are production core. `project_context` must return README context, notebook filenames, and file docstrings; `file_context` must include imports, header docs, Markdown, and definition summaries with regex filters; `chapter_context` must include the notebook head plus one selected section; and `symbol_context` must explain one implementation with nearby docs, examples/tests, callers, and depth-controlled callees.

In [ ]:
#| hide
from contextlib import redirect_stdout
from io import StringIO
from fastcore.nbio import mk_cell, new_nb, read_nb, write_nb
from fastcore.test import test_eq
from nbskill.read import chapter_context as _example_chapter_context
from nbskill.read import file_context as _example_file_context
from nbskill.read import project_context as _example_project_context
from nbskill.read import symbol_context as _example_symbol_context
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook, write_tool_notebook

In [ ]:
with write_demo_notebook("01_read_example.ipynb") as path:
    write_nb(new_nb([
        mk_cell("# Reader\nFile preamble.", cell_type="markdown"),
        mk_cell("preamble = True", cell_type="code"),
        mk_cell("## Demo", cell_type="markdown"),
        mk_cell("#| export\ndef answer():\n    \"\"\"Return the demo answer.\"\"\"\n    return 42", cell_type="code"),
        mk_cell("This explains why the demo answer exists.", cell_type="markdown"),
        mk_cell("answer()", cell_type="code"),
        mk_cell("assert answer() == 42", cell_type="code"),
    ]), path)
    print("project_context")
    _example_project_context(str(path))
    print("\nfile_context")
    _example_file_context(str(path), include_re="answer")
    print("\nchapter_context")
    _example_chapter_context(str(path), name="Demo")
    print("\nsymbol_context")
    _example_symbol_context(str(path), "answer", depth=0)


In [ ]:
#| export
import ast
import copy
import json
import os
import re
import shlex
from pathlib import Path

from fastcore.nbio import read_nb
from fastcore.basics import patch

from nbskill.foundation import (
    Notebook, NotebookCell, call_name, cell_matches_type, cell_output_text, cell_prefix,
    cell_source, chapter_index_set, chapter_spans, find_cell_by_id, first_line,
    heading_title, is_exported_code_cell, matches_filter, notebook_paths,
    source_hash, source_without_directives, symbol_short_name,
)

#### Output shapes

The context readers serve four attention levels. `project_context` maps the repository; `file_context` shows a notebook's cells, Markdown, imports, and definitions; `chapter_context` shows the notebook head plus one chapter in source order; and `symbol_context` focuses one implementation followed by its trailing Docs, examples/tests, callers, and callees.

Use `context` for triage and these focused readers when you already know which surface you need. The structured return values keep cell ids and indexes so an edit can target the exact source cell.

In [ ]:
#| export
def _format_overview(items, show_ids=False):
    lines = []
    for idx, cell in items:
        summary = first_line(cell.source)
        lines.append(f"{cell_prefix(idx, cell, show_ids)} | {summary}")
    return "\n".join(lines)

In [ ]:
#| export
def _definition_lines(node, indent=""):
    tmp = copy.deepcopy(node)
    tmp.body = [ast.Pass()]
    ast.fix_missing_locations(tmp)
    lines = []
    for line in ast.unparse(tmp).splitlines():
        if line.strip() == "pass": continue
        lines.append(f"{indent}{line}" if line else line)
    return lines

In [ ]:
#| export
def _docstring_lines(node, indent="    "):
    doc = ast.get_docstring(node)
    if not doc: return []
    lines = doc.splitlines()
    quote = chr(34) * 3
    if len(lines) == 1: return [f"{indent}{quote}{lines[0]}{quote}"]
    return [indent + quote, *[f"{indent}{line}" for line in lines], indent + quote]

In [ ]:
#| export
def _function_overview(node, indent=""):
    return [*_definition_lines(node, indent=indent), *_docstring_lines(node, indent=indent + "    ")]

In [ ]:
#| export
def _class_overview(node):
    lines = [*_definition_lines(node), *_docstring_lines(node)]
    methods = [child for child in node.body if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef))]
    for method in methods:
        if lines and lines[-1] != "": lines.append("")
        lines += _function_overview(method, indent="    ")
    return lines

In [ ]:
#| export
def _code_overview(cell):
    try: tree = ast.parse(cell.source)
    except SyntaxError: return []
    lines = []
    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom)): lines.append(ast.unparse(node))
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): lines += _function_overview(node)
        elif isinstance(node, ast.ClassDef): lines += _class_overview(node)
        if lines and lines[-1] != "": lines.append("")
    if lines and lines[-1] == "": lines.pop()
    return lines

In [ ]:
#| export
def _format_source(source, line_numbers=False):
    if not line_numbers: return source
    lines = source.splitlines() or [""]
    return "\n".join(f"{idx} | {line}" for idx, line in enumerate(lines, start=1))

In [ ]:
#| export
def _format_full(items, show_ids=False, line_numbers=False):
    chunks = []
    for idx, cell in items:
        chunks.append(f"{cell_prefix(idx, cell, show_ids)}\n{_format_source(cell_source(cell), line_numbers=line_numbers)}")
    return "\n\n".join(chunks)

## Query and selection

Selectors turn short human queries into stable notebook, chapter, cell, and symbol matches.

#### A small query language

Automation needs stable selectors, but humans need short commands. The query helpers accept aliases like `id`, `type`, `chapter`, and `contains`, then normalize them into one internal selection shape.

In [ ]:
#| export
_QUERY_KEY_ALIASES = {
    "id": "cell_id",
    "cell": "cell_id",
    "cell_id": "cell_id",
    "chapter": "chapter",
    "section": "chapter",
    "type": "cell_type",
    "class": "cell_type",
    "cell_type": "cell_type",
    "contains": "contains",
    "text": "contains",
    "regex": "regex",
    "re": "regex",
    "error": "has_error",
    "errors": "has_error",
    "has_error": "has_error",
    "export": "export",
    "exports": "export",
    "exported": "export",
    "header": "header",
    "headers": "header",
    "heading": "header",
    "headings": "header",
    "outline": "header",
}

In [ ]:
#| export
def _normalize_query_key(key):
    name = _QUERY_KEY_ALIASES.get(str(key).strip().lower())
    if name is None:
        choices = ", ".join(sorted(_QUERY_KEY_ALIASES))
        raise ValueError(f"Unknown query key {key!r}; use one of: {choices}")
    return name

In [ ]:
#| export
def _normalize_query_dict(spec):
    return {_normalize_query_key(key): None if value is None else str(value) for key, value in dict(spec).items()}

In [ ]:
#| export
def _parse_query_terms(text):
    spec, bare = {}, []
    for term in shlex.split(str(text)):
        sep = "=" if "=" in term else ":" if ":" in term else None
        if sep is None:
            key = _QUERY_KEY_ALIASES.get(term.strip().lower())
            if key in {"has_error", "export", "header"}: spec[key] = "true"
            else: bare.append(term)
            continue
        key, value = term.split(sep, 1)
        spec[_normalize_query_key(key)] = value
    if bare and "contains" not in spec: spec["contains"] = " ".join(bare)
    return spec

In [ ]:
#| export
def _parse_query(query):
    if query is None: return [{}]
    if isinstance(query, dict): return [_normalize_query_dict(query)]
    if isinstance(query, (list, tuple)):
        specs = []
        for item in query: specs.extend(_parse_query(item))
        return specs or [{}]

    text = str(query).strip()
    if not text: return [{}]
    try: parsed = json.loads(text)
    except json.JSONDecodeError:
        return [_parse_query_terms(part) for part in text.split(";") if part.strip()]
    return _parse_query(parsed)

In [ ]:
#| export
def _query_bool(value, default=True):
    if value is None: return default
    return str(value).strip().lower() not in {"0", "false", "no", "off", "none"}

In [ ]:
#| export
def _output_get(output, key, default=None):
    return output.get(key, default) if isinstance(output, dict) else getattr(output, key, default)

In [ ]:
#| export
def _cell_has_error(cell):
    return any(_output_get(output, "output_type") == "error" for output in (getattr(cell, "outputs", []) or []))

In [ ]:
#| export
def _cell_header_level(cell):
    if getattr(cell, "cell_type", None) != "markdown": return None
    line = first_line(cell_source(cell)).strip()
    if not line.startswith("#"): return None
    return len(line) - len(line.lstrip("#"))

In [ ]:
#| export
def _cell_matches_header(cell, value):
    level = _cell_header_level(cell)
    if level is None: return False
    text = str(value or "").strip()
    return level == int(text) if text.isdigit() else _query_bool(value)

In [ ]:
#| export
def _query_chapter_index_set(nb, name):
    span = _one_chapter_span(nb.cells, name)
    return set(range(span["start"], span["end"]))

In [ ]:
#| export
def _select_query_items(nb, spec):
    items = [find_cell_by_id(nb.cells, spec["cell_id"])] if spec.get("cell_id") else list(enumerate(nb.cells))
    if spec.get("chapter") is not None:
        chapter_idxs = _query_chapter_index_set(nb, spec["chapter"])
        items = [(i, c) for i, c in items if i in chapter_idxs]
    if spec.get("cell_type") is not None: items = [(i, c) for i, c in items if cell_matches_type(c, spec["cell_type"])]
    if spec.get("header") is not None: items = [(i, c) for i, c in items if _cell_matches_header(c, spec["header"])]
    if spec.get("export") is not None: items = [(i, c) for i, c in items if is_exported_code_cell(c) == _query_bool(spec["export"])]
    if spec.get("has_error") is not None: items = [(i, c) for i, c in items if _cell_has_error(c) == _query_bool(spec["has_error"])]
    if spec.get("contains") is not None: items = [(i, c) for i, c in items if spec["contains"] in cell_source(c)]
    if spec.get("regex") is not None: items = [(i, c) for i, c in items if matches_filter(cell_source(c), spec["regex"])]
    return items

In [ ]:
#| export
def _context_result(kind, text, verbose=True, **data):
    result = {"kind": kind, "text": text, **data}
    if verbose and text: print(text)
    return result

In [ ]:
#| export
def _context_root(path):
    start = Path(path).expanduser()
    if start.suffix == ".ipynb": start = start.parent
    if not start.exists() and start.suffix: start = start.parent
    start = start.resolve() if start.exists() else (Path.cwd() / start).resolve()
    for item in [start, *start.parents]:
        if (item / "README.md").exists(): return item
    return start

In [ ]:
#| export
def _readme_context(root, max_sections=3):
    readme = Path(root) / "README.md"
    if not readme.exists(): return []
    sections, current = [], []
    for line in readme.read_text(encoding="utf-8", errors="ignore").splitlines():
        if re.match(r"^#{1,2}\s+", line) and current:
            sections.append("\n".join(current).strip())
            current = []
        if line.strip() or current: current.append(line)
    if current: sections.append("\n".join(current).strip())
    return [section for section in sections if section][:max_sections]

In [ ]:
#| export
def _notebook_header_items(cells):
    items, started = [], False
    for idx, cell in enumerate(cells):
        if getattr(cell, "cell_type", None) != "markdown":
            if started: break
            continue
        source = cell_source(cell).strip()
        is_h1 = bool(re.search(r"^#\s+", source, flags=re.MULTILINE))
        is_h2 = bool(re.search(r"^##\s+", source, flags=re.MULTILINE))
        if is_h2 and started: break
        if is_h1 or is_h2: started = True
        if started: items.append((idx, cell))
    return items

In [ ]:
#| export
def _notebook_docstring(path):
    nb = read_nb(path)
    items = _notebook_header_items(nb.cells)
    return {
        "path": str(path),
        "cells": [
            {"cell_id": getattr(cell, "id", ""), "source": cell_source(cell).strip()}
            for _, cell in items if cell_source(cell).strip()
        ],
    }

In [ ]:
#| export
def _context_match(text, include_re=None, exclude_re=None):
    text = str(text or "")
    if include_re and not re.search(include_re, text, flags=re.MULTILINE): return False
    if exclude_re and re.search(exclude_re, text, flags=re.MULTILINE): return False
    return True

In [ ]:
#| export
def _format_context_blocks(title, blocks):
    lines = [title]
    for heading, body in blocks:
        if not body: continue
        lines.extend(["", heading])
        if isinstance(body, str): lines.append(body)
        else: lines.extend(body)
    return "\n".join(lines).rstrip()

In [ ]:
#| export
def _import_lines(cell):
    if getattr(cell, "cell_type", None) != "code": return []
    try: tree = ast.parse(source_without_directives(cell_source(cell)))
    except SyntaxError: return []
    return [ast.unparse(node) for node in tree.body if isinstance(node, (ast.Import, ast.ImportFrom))]

In [ ]:
#| export
def _definition_record(path, idx, cell, node, symbol=None, kind=None):
    symbol = symbol or getattr(node, "name", "")
    kind = kind or ("class" if isinstance(node, ast.ClassDef) else "function")
    lines = _class_overview(node) if isinstance(node, ast.ClassDef) else _function_overview(node)
    return {
        "path": str(path),
        "cell_id": getattr(cell, "id", ""),
        "cell_idx": idx,
        "symbol": symbol,
        "kind": kind,
        "text": "\n".join(lines),
    }

In [ ]:
#| export
def _definition_records(path, nb):
    records = []
    for idx, cell in enumerate(nb.cells):
        if getattr(cell, "cell_type", None) != "code": continue
        try: tree = ast.parse(source_without_directives(cell_source(cell)))
        except SyntaxError: continue
        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                records.append(_definition_record(path, idx, cell, node))
    return records

In [ ]:
#| export
def _source_for_node(cell, node):
    source = source_without_directives(cell_source(cell))
    segment = ast.get_source_segment(source, node)
    if segment: return segment.strip()
    lines = source.splitlines()
    start = max(getattr(node, "lineno", 1) - 1, 0)
    end = getattr(node, "end_lineno", start + 1)
    return chr(10).join(lines[start:end]).strip()

In [ ]:
#| export
def _call_matches_context_symbol(name, symbol):
    name = str(name or "")
    short = symbol_short_name(symbol)
    return name == symbol or name.rsplit(".", 1)[-1] == short

In [ ]:
#| export
def _cell_calls_symbol(cell, symbol):
    if getattr(cell, "cell_type", None) != "code": return False
    try: tree = ast.parse(source_without_directives(cell_source(cell)))
    except SyntaxError: return False
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and _call_matches_context_symbol(call_name(node.func), symbol): return True
    return False

In [ ]:
#| export
def _example_test_records(nb, symbol, definition_idx=None):
    items = _following_examples(nb.cells, definition_idx, len(nb.cells)) if definition_idx is not None else list(enumerate(nb.cells))
    records = []
    for idx, cell in items:
        if not _cell_calls_symbol(cell, symbol): continue
        kind = "test" if cell_matches_type(cell, "test") else "example" if cell_matches_type(cell, "example") else "usage"
        records.append({
            "cell_id": getattr(cell, "id", ""),
            "cell_idx": idx,
            "kind": kind,
            "source": cell_source(cell).strip(),
            "output": cell_output_text(cell),
        })
    return records

In [ ]:
#| export
def _symbol_location(path, nb, symbol):
    idx = _find_symbol_cell(nb, symbol)
    cell = nb.cells[idx]
    node = _find_symbol_node(cell, symbol)
    return idx, cell, node

In [ ]:
#| export
def _callee_summary(path, symbol, depth, seen=None, graph_data=None):
    if depth <= 0: return []
    seen = set() if seen is None else seen
    if symbol in seen: return []
    seen.add(symbol)
    try:
        from nbskill.graph import symbol_graph_data
        data = graph_data or symbol_graph_data(path, symbol)
    except Exception:
        return []
    items = []
    for callee in data.get("callees", []):
        if callee in seen: continue
        definitions = data.get("graph", {}).get("definitions", [])
        definition = next((record for record in definitions if record.get("symbol") == callee), None)
        summary = ""
        if definition:
            try:
                callee_nb = read_nb(definition["path"])
                callee_cell = callee_nb.cells[definition["cell_idx"]]
                summary = "\n".join(_symbol_signature_text(callee_cell, callee))
            except Exception:
                summary = ""
        item = {
            "symbol": callee,
            "path": definition.get("path") if definition else "",
            "cell_id": definition.get("cell_id") if definition else "",
            "summary": summary,
            "callees": _callee_summary(path, callee, depth - 1, seen),
        }
        items.append(item)
    return items

In [ ]:
#| export
def _format_callee_items(items, indent=""):
    lines = []
    for item in items:
        loc = f" {item['path']} id={item['cell_id']}" if item.get("path") else ""
        lines.append(f"{indent}- {item['symbol']}:{loc}".rstrip())
        if item.get("summary"):
            lines.extend(f"{indent}  {line}" for line in item["summary"].splitlines())
        lines.extend(_format_callee_items(item.get("callees", []), indent=indent + "  "))
    return lines

## Public notebook context

This is the main notebook-facing surface: it combines the smaller readers and returns a structured result while optionally printing the rendered text.

#### The public notebook reader
The preferred public reader is `context(target, scope=".")`. `target` can be `project`, a notebook path, a cell id, a chapter title, a Python symbol, or a literal string to search for when no structured target resolves; `scope` narrows where notebook targets and fallback file searches run.
The older focused readers remain as small internal building blocks, but the simple path is one target plus one optional search scope.

In [ ]:
#| export
def _normalize_chapter_title(value):
    return re.sub(r"\s+", " ", str(value).strip().lower())

In [ ]:
#| export
def _chapter_spans_for_nb(cells): return chapter_spans(cells, levels=(2,), fallback="Notebook")

In [ ]:
#| export
def _all_heading_spans_for_nb(cells): return chapter_spans(cells, levels=range(1, 7), fallback="Notebook")

In [ ]:
#| export
def _notebook_head_items(cells):
    spans = _chapter_spans_for_nb(cells)
    head_end = spans[0]["start"] if spans else len(cells)
    return [(idx, cells[idx]) for idx in range(head_end)]

In [ ]:
#| export
def _chapter_span_for_index(cells, idx):
    spans = _chapter_spans_for_nb(cells)
    if spans and idx < spans[0]["start"]:
        return dict(title="Notebook head", start=0, end=spans[0]["start"])
    for span in spans:
        if span["start"] <= idx < span["end"]: return span
    raise ValueError(f"Cell index {idx} is outside the notebook")

In [ ]:
#| export
def _chapter_matches(span, name):
    title = span["title"]
    if matches_filter(title, name): return True
    wanted, candidate = _normalize_chapter_title(name), _normalize_chapter_title(title)
    return bool(wanted and (wanted in candidate or candidate in wanted))

In [ ]:
#| export
def _one_chapter_span(cells, name):
    spans = _chapter_spans_for_nb(cells)
    matches = [span for span in spans if _chapter_matches(span, name)]
    if not matches:
        nested_spans = _all_heading_spans_for_nb(cells)
        nested_matches = [span for span in nested_spans if _chapter_matches(span, name)]
        if nested_matches: spans, matches = nested_spans, nested_matches
    if len(matches) == 1: return matches[0]
    titles = ", ".join(span["title"] for span in spans[:8]) or "none"
    if not matches: raise ValueError(f"No chapter matches {name!r}. Available chapters: {titles}")
    matched = ", ".join(span["title"] for span in matches)
    raise ValueError(f"Chapter {name!r} matches multiple chapters: {matched}")

In [ ]:
#| export
def _query_items(nb, query):
    items, seen = [], set()
    for spec in _parse_query(query):
        for idx, cell in _select_query_items(nb, spec):
            if idx in seen: continue
            seen.add(idx)
            items.append((idx, cell))
    return items

In [ ]:
#| export
def _selected_chapter_span(nb, query=None, name=None, any_cell_id=None):
    selectors = [value is not None for value in (query, name, any_cell_id)]
    if sum(selectors) != 1: raise ValueError("Pass exactly one of query, name, or any_cell_id")
    if name is not None: return _one_chapter_span(nb.cells, name)
    if any_cell_id is not None:
        idx, _ = find_cell_by_id(nb.cells, any_cell_id)
        return _chapter_span_for_index(nb.cells, idx)
    items = _query_items(nb, query)
    if not items: raise ValueError(f"No cells match query {query!r}")
    return _chapter_span_for_index(nb.cells, items[0][0])

In [ ]:
#| export
def _chapter_items(nb, span):
    idxs = set()
    items = []
    for idx, cell in [*_notebook_head_items(nb.cells), *[(i, nb.cells[i]) for i in range(span["start"], span["end"])]]:
        if idx in idxs: continue
        idxs.add(idx)
        items.append((idx, cell))
    return items

In [ ]:
#| export
def _chapter_intro_markdown_items(cells, span):
    items = []
    for idx in range(span["start"], span["end"]):
        cell = cells[idx]
        if getattr(cell, "cell_type", None) != "markdown": continue
        source = cell_source(cell)
        level = _markdown_heading_level(source)
        if idx != span["start"] and level in (3, 4): break
        if source.strip(): items.append((idx, cell))
    return items

In [ ]:
#| export
def _chapter_intro_items(nb, span):
    idxs = set()
    items = []
    for idx, cell in [*_notebook_head_items(nb.cells), *_chapter_intro_markdown_items(nb.cells, span)]:
        if idx in idxs: continue
        idxs.add(idx)
        items.append((idx, cell))
    return items

In [ ]:
#| export
def _filter_chapter_items(items, cell_type=None, include_re=None, exclude_re=None):
    filtered = []
    for idx, cell in items:
        source = cell_source(cell)
        if cell_type is not None and not cell_matches_type(cell, cell_type): continue
        if not _context_match(source, include_re, exclude_re): continue
        filtered.append((idx, cell))
    return filtered

In [ ]:
#| export
def project_context(
    path: str = ".",  # Project directory, notebook path, or notebook glob root
    verbose: bool = True,  # Print rendered context; pass False to only return structured data
):
    "Show README sections, notebook filenames, and notebook file docstrings."
    root = _context_root(path)
    notebooks = [str(item) for item in notebook_paths(path)]
    docstrings = [_notebook_docstring(item) for item in notebooks]
    readme_sections = _readme_context(root)
    nl = chr(10)
    def _first_line(src): return src.strip().splitlines()[0].lstrip("#").strip() if src.strip() else ""
    def _nb_summary(item):
        cells = item["cells"]
        if not cells: return None
        title = _first_line(cells[0]["source"])
        tagline = _first_line(cells[1]["source"]) if len(cells) > 1 else ""
        name = Path(item["path"]).name
        return f"- {name}: {title}" + (f" — {tagline}" if tagline else "")
    summaries = [s for item in docstrings for s in [_nb_summary(item)] if s]
    blocks = [
        ("Notebooks", summaries),
        ("README", (nl * 2).join(readme_sections)),
    ]
    text = _format_context_blocks(f"Project context: {root}{nl}Requested path: {path}", blocks)
    return _context_result(
        "project_context", text, verbose=verbose, root=str(root), requested_path=str(path),
        readme_sections=readme_sections, notebooks=notebooks, docstrings=docstrings,
    )

In [ ]:
#| export
def chapter_context(
    path: str,  # Notebook path
    query: str | None = None,  # Query for any cell inside the chapter
    name: str | None = None,  # Chapter title string or regex
    any_cell_id: str | None = None,  # Any cell id inside the chapter
    verbose: bool = True,  # Print rendered context; pass False to only return structured data
    overview: bool = False,  # Show only notebook head and chapter intro markdown
    cell_type: str | None = None,  # Optional cell type or semantic type filter for deep dives
    include_re: str | None = None,  # Optional regex that matched cell source must include
    exclude_re: str | None = None,  # Optional regex that matched cell source must not include
):
    "Show one chapter in source order, with optional overview and cell filters."
    nb = read_nb(path)
    span = _selected_chapter_span(nb, query=query, name=name, any_cell_id=any_cell_id)
    items = _chapter_intro_items(nb, span) if overview else _chapter_items(nb, span)
    items = _filter_chapter_items(items, cell_type=cell_type, include_re=include_re, exclude_re=exclude_re)
    text = _format_full(items, show_ids=True, line_numbers=False)
    cells = [{"cell_id": getattr(cell, "id", ""), "cell_idx": idx, "cell_type": cell.cell_type, "source": cell_source(cell)} for idx, cell in items]
    return _context_result(
        "chapter_context", text, verbose=verbose, path=str(path), query=query,
        name=name, any_cell_id=any_cell_id, overview=overview, cell_type=cell_type,
        include_re=include_re, exclude_re=exclude_re, chapter=span, cells=cells,
    )

## Python source context

Notebook work often crosses into generated modules. The AST readers keep that handoff compact and navigable.

Python files get a compact AST-backed reader too. `python_file_context` summarizes public definitions without dumping full source, while `python_symbol_context` opens one symbol and shows the local callers and callees that matter for an edit.

In [ ]:
#| export
def _python_ast(path):
    path = Path(path)
    source = path.read_text(encoding="utf-8")
    return source, ast.parse(source, filename=str(path))

In [ ]:
#| export
def _python_doc_first_line(node):
    doc = ast.get_docstring(node) or ""
    return doc.strip().splitlines()[0].strip() if doc.strip() else ""

In [ ]:
#| export
def _python_source_segment(source, node):
    return (ast.get_source_segment(source, node) or "").strip()

In [ ]:
#| export
def _python_signature(source, node):
    for line in _python_source_segment(source, node).splitlines():
        line = line.strip()
        if line and not line.startswith("@"): return line
    return ""

In [ ]:
#| export
def _python_call_name(node):
    if isinstance(node, ast.Name): return node.id
    if isinstance(node, ast.Attribute):
        prefix = _python_call_name(node.value)
        return f"{prefix}.{node.attr}" if prefix and prefix != "self" else node.attr
    return ""

In [ ]:
#| export
def _python_calls(node):
    return sorted({name for name in (_python_call_name(item.func) for item in ast.walk(node) if isinstance(item, ast.Call)) if name})

In [ ]:
#| export
def _python_assign_names(node):
    targets = [node.target] if isinstance(node, ast.AnnAssign) else getattr(node, "targets", [])
    return [target.id for target in targets if isinstance(target, ast.Name)]

In [ ]:
#| export
def _python_definition_records(path):
    path, records = Path(path), []
    source, tree = _python_ast(path)

    def add_record(node, symbol, kind):
        public = not any(part.startswith("_") for part in symbol.split("."))
        records.append(dict(
            path=str(path), symbol=symbol, kind=kind, public=public,
            line=getattr(node, "lineno", 1), end_line=getattr(node, "end_lineno", getattr(node, "lineno", 1)),
            signature=_python_signature(source, node), docstring=_python_doc_first_line(node),
            source=_python_source_segment(source, node), calls=_python_calls(node),
        ))

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            add_record(node, node.name, "async function" if isinstance(node, ast.AsyncFunctionDef) else "function")
        elif isinstance(node, ast.ClassDef):
            add_record(node, node.name, "class")
            for child in node.body:
                if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    add_record(child, f"{node.name}.{child.name}", "method")
        elif isinstance(node, (ast.Assign, ast.AnnAssign)):
            for name in _python_assign_names(node):
                if not name.startswith("_"):
                    records.append(dict(
                        path=str(path), symbol=name, kind="assignment", public=True,
                        line=getattr(node, "lineno", 1), end_line=getattr(node, "end_lineno", getattr(node, "lineno", 1)),
                        signature=_python_signature(source, node), docstring="", source=_python_source_segment(source, node), calls=_python_calls(node),
                    ))
    return records

In [ ]:
#| export
def _python_public_symbols(records):
    return [item for item in records if item.get("public")]

In [ ]:
#| export
def _python_symbol_record(path, symbol):
    symbol = str(symbol)
    records = _python_definition_records(path)
    for item in records:
        if item["symbol"] == symbol: return item, records
    matches = [item for item in records if item["symbol"].split(".")[-1] == symbol]
    if len(matches) == 1: return matches[0], records
    return None, records

In [ ]:
#| export
def _python_symbol_callers(records, symbol):
    short = str(symbol).split(".")[-1]
    callers = []
    for item in records:
        if item["symbol"] == symbol or item.get("kind") == "class": continue
        calls = {call.split(".")[-1] for call in item.get("calls", [])}
        if short in calls or symbol in item.get("calls", []): callers.append(item)
    return callers

In [ ]:
#| export
def _format_python_summary(records):
    lines = []
    for item in records:
        if item["kind"] not in {"function", "async function", "class", "method"}: continue
        doc = f": {item['docstring']}" if item.get("docstring") else ""
        lines.append(f"- {item['symbol']} ({item['kind']}, line {item['line']}){doc}")
    return lines

In [ ]:
#| export
def _format_python_symbols(records):
    return [f"- {item['symbol']} ({item['kind']}, line {item['line']})" for item in _python_public_symbols(records)]

In [ ]:
#| export
def python_file_context(
    path: str,  # Python file path
    include_private: bool = False,  # Include private definitions in the summary
    verbose: bool = True,  # Print rendered context; pass False to only return structured data
):
    "Show a compact AST summary for one Python file."
    records = _python_definition_records(path)
    shown = records if include_private else _python_public_symbols(records)
    text = _format_context_blocks(
        f"Python file context: {path}",
        [("Summary", _format_python_summary(shown)), ("Public symbols", _format_python_symbols(records))],
    )
    return _context_result("python_file_context", text, verbose=verbose, path=str(path), definitions=records, public_symbols=_python_public_symbols(records))

In [ ]:
#| export
def _python_trim_source(source, limit=1200):
    source = str(source or "")
    return source if len(source) <= limit else f"{source[:limit].rstrip()}\n... <truncated>"

In [ ]:
#| export
def python_symbol_context(
    path: str,  # Python file path
    symbol: str,  # Function, class, method, or assignment symbol to inspect
    verbose: bool = True,  # Print rendered context; pass False to only return structured data
):
    "Show source, local callers, and callees for one Python-file symbol."
    item, records = _python_symbol_record(path, symbol)
    if item is None:
        candidates = [record["symbol"] for record in _python_public_symbols(records)]
        exc = ValueError(f"No Python symbol {symbol!r} in {path}")
        exc.candidates = candidates
        raise exc
    callers = _python_symbol_callers(records, item["symbol"])
    callees = item.get("calls", [])
    text = _format_context_blocks(
        f"Python symbol context: {item['symbol']}\nLocation: {path}:{item['line']}",
        [
            ("Implementation", _python_trim_source(item.get("source", ""))),
            ("Callers", [f"- {caller['symbol']} ({caller['kind']}, line {caller['line']})" for caller in callers]),
            ("Callees", [f"- {callee}" for callee in callees]),
        ],
    )
    return _context_result(
        "python_symbol_context", text, verbose=verbose, path=str(path), symbol=item["symbol"],
        location=dict(line=item["line"], end_line=item["end_line"]), source=item.get("source", ""),
        callers=callers, callees=callees, symbols=[item["symbol"]], definitions=records,
    )

In [ ]:
#| export
def _context_existing_python(target):
    raw = str(target or "")
    if not raw: return None
    path = Path(raw).expanduser()
    candidates = [path] if path.is_absolute() else [path, Path.cwd() / path]
    for candidate in candidates:
        if candidate.exists() and candidate.is_file() and candidate.suffix == ".py": return candidate
    return None

In [ ]:
#| export
def _context_python_qualified_payload(target):
    text = str(target or "")
    if "#" not in text: return None
    path_text, ref = text.rsplit("#", 1)
    path = _context_existing_python(path_text)
    if path is None: return None
    if not ref: return python_file_context(str(path), verbose=False), "python_file", {}
    return python_symbol_context(str(path), ref, verbose=False), "python_symbol", {}

In [ ]:
#| export
def _context_python_scope_payload(target, scope):
    path = _context_existing_python(scope)
    if path is None: return None
    if target in {str(path), path.name}: return python_file_context(str(path), verbose=False), "python_file", {}
    item, _ = _python_symbol_record(path, target)
    if item is None: return None
    return python_symbol_context(str(path), item["symbol"], verbose=False), "python_symbol", {}

In [ ]:
with write_demo_notebook("01_read_python_context_example.py") as path:
    path.write_text(
        "PUBLIC_VALUE = 1\n\n"
        "def helper(value):\n"
        "    \"\"\"Prepare a value.\n\n    Hidden detail.\"\"\"\n"
        "    return value + PUBLIC_VALUE\n\n"
        "def add(a, b):\n"
        "    \"\"\"Add two values.\n\n    Hidden detail.\"\"\"\n"
        "    return helper(a) + b\n\n"
        "def uses_add(value):\n"
        "    return add(value, 1)\n",
        encoding="utf-8",
    )
    py_file = python_file_context(str(path), verbose=False)
    py_symbol = python_symbol_context(str(path), "add", verbose=False)
py_file["public_symbols"][0]["symbol"], py_file["definitions"][1]["docstring"], py_symbol["callers"][0]["symbol"], py_symbol["callees"]

In [ ]:
#| export
def file_context(
    path: str,  # Notebook or Python file path
    include_re: str | None = None,  # Optional regex for markdown and definition items to include
    exclude_re: str | None = None,  # Optional regex for markdown and definition items to exclude
    verbose: bool = True,  # Print rendered context; pass False to only return structured data
):
    "Show compact context for one notebook or Python file; notebook cells stay in source order."
    if str(path).endswith(".py"):
        return python_file_context(path, verbose=verbose)
    nb = read_nb(path)
    imports = []
    for _, cell in enumerate(nb.cells): imports.extend(_import_lines(cell))
    imports = list(dict.fromkeys(imports))
    header = [{"cell_id": getattr(cell, "id", ""), "cell_idx": idx, "source": cell_source(cell).strip()} for idx, cell in _notebook_header_items(nb.cells)]
    markdown = [
        {"cell_id": getattr(cell, "id", ""), "cell_idx": idx, "source": cell_source(cell).strip()}
        for idx, cell in enumerate(nb.cells)
        if getattr(cell, "cell_type", None) == "markdown" and _context_match(cell_source(cell), include_re, exclude_re)
    ]
    definitions = [item for item in _definition_records(path, nb) if _context_match(f"{item['symbol']}\n{item['text']}", include_re, exclude_re)]
    blocks = [
        ("Header", [item["source"] for item in header]),
        ("Imports", imports),
        ("Markdown", [f"Cell id={item['cell_id']}\n{item['source']}" for item in markdown]),
        ("Definitions", [f"Cell id={item['cell_id']} {item['kind']} {item['symbol']}\n{item['text']}" for item in definitions]),
    ]
    text = _format_context_blocks(f"File context: {path}", blocks)
    return _context_result(
        "file_context", text, verbose=verbose, path=str(path), include_re=include_re,
        exclude_re=exclude_re, imports=imports, header=header, markdown=markdown, definitions=definitions,
    )


Filtered context is the project-wide escape hatch for questions that are too broad for one target but still need notebook-aware cells instead of raw JSON. It searches notebooks in `scope`, keeps stable cell ids in the result, and caps both the number of matches and each source snippet so agents can ask broad questions without flooding their context.

`filter_context` is the search-shaped counterpart to `context`: it walks a notebook scope in source order and returns the cells whose source matches query selectors and include/exclude regexes.

Use `query="type=export"` to find implementations, `query="chapter=Name"` to read one section, or `query="regex=term"` with `before` and `after` to inspect the local code → Docs → example/test neighborhood. The result keeps notebook paths, cell ids, cell indexes, semantic types, and source text so an agent can follow up with a focused `context` call instead of reading raw notebook JSON.

In [ ]:
#| export
_CONTEXT_SEARCH_SKIP_DIRS = set(".git .hg .svn .venv venv env __pycache__ .ipynb_checkpoints .pytest_cache .mypy_cache .ruff_cache .quarto .tox .nox node_modules dist build".split())
_CONTEXT_SEARCH_SKIP_SUFFIXES = set(".ipynb .pyc .pyo .so .dylib .dll .png .jpg .jpeg .gif .webp .ico .pdf .zip .tar .gz .bz2 .xz .7z .sqlite .db .jsonl .log".split())
_CONTEXT_CELL_SOURCE_CHARS = 50000
_CONTEXT_CELL_OUTPUT_CHARS = 4000
_CONTEXT_SEARCH_MAX_FILES = 2000
_CONTEXT_SEARCH_MAX_BYTES = 1_000_000


def _trim_context_source(source, limit=1200):
    source = str(source or "").strip()
    if limit is None or limit <= 0 or len(source) <= limit: return source
    return source[:limit].rstrip() + f"\n... {len(source) - limit} chars omitted ..."

In [ ]:
#| export
def _context_notebooks(scope):
    notebooks = [str(item) for item in notebook_paths(scope or ".")]
    if not notebooks: raise ValueError(f"No notebooks found in scope {scope!r}")
    return notebooks

In [ ]:
#| export
def _context_existing_notebook(target):
    path = Path(str(target)).expanduser()
    if path.suffix == ".ipynb" and path.exists(): return str(path)
    return None

In [ ]:
#| export
def _context_named_notebook(target, notebooks):
    wanted = str(target)
    matches = []
    for path in notebooks:
        nb_path = Path(path)
        if wanted in {str(nb_path), nb_path.name, nb_path.stem}: matches.append(path)
    if len(matches) > 1:
        shown = "\n".join(f"- {item}" for item in matches[:8])
        raise ValueError(f"Notebook target {target!r} matched multiple notebooks:\n{shown}")
    return matches[0] if matches else None

In [ ]:
#| export
_FILTER_CONTEXT_VIEWS = {"source", "summary", "cell"}

In [ ]:
#| export
def _filter_context_view(view):
    view = "source" if view in (None, "", "cells") else str(view).lower()
    if view not in _FILTER_CONTEXT_VIEWS:
        raise ValueError(f"view must be one of {sorted(_FILTER_CONTEXT_VIEWS)}, not {view!r}")
    return view

In [ ]:
#| export
def _filter_cell_summary(path, item):
    source = item.get("source", "")
    first = first_line(source).strip()
    if not first: first = "(empty)"
    error = item.get("error", "")
    suffix = f" error={error}" if error else ""
    return f"{path}#{item['cell_id']} idx={item['cell_idx']} type={item['cell_type']} {item['semantic_type']}{suffix} | {first}"

In [ ]:
#| export
def _filter_cell_error(cell):
    for output in getattr(cell, "outputs", []) or []:
        if _output_get(output, "output_type") == "error":
            ename = _output_get(output, "ename", "Error")
            evalue = _output_get(output, "evalue", "")
            return f"{ename}: {evalue}".rstrip(": ")
    return ""

In [ ]:
#| export
def _filter_match_record(path, idx, cell, max_chars_per_cell):
    item = NotebookCell(cell, idx=idx).context_record()
    item["source"] = _trim_context_source(item.get("source", ""), max_chars_per_cell)
    item["output"] = _trim_context_source(item.get("output", ""), _CONTEXT_CELL_OUTPUT_CHARS)
    item.update(path=str(path), error=_filter_cell_error(cell))
    item["summary"] = _filter_cell_summary(path, item)
    return item

In [ ]:
#| export
def _neighbor_records(path, nb, idx, before=0, after=0):
    start = max(0, idx - max(0, int(before or 0)))
    end = min(len(nb.cells), idx + max(0, int(after or 0)) + 1)
    return [_filter_match_record(path, pos, nb.cells[pos], 0) for pos in range(start, end) if pos != idx]

In [ ]:
#| export
def _render_filter_source(item, line_numbers=False):
    nl = chr(10)
    header = f"{item['path']} id={item['cell_id']} idx={item['cell_idx']} type={item['cell_type']}"
    source = _format_source(item.get("source", ""), line_numbers=line_numbers)
    text = f"{header}{nl}{source}".rstrip()
    if item.get("output"): text = f"{text}{nl}Output:{nl}{item['output']}"
    return text

In [ ]:
#| export
def _render_filter_match(item, view="source", line_numbers=False):
    if view == "summary": return item["summary"]
    text = _render_filter_source(item, line_numbers=(line_numbers or view == "cell"))
    if item.get("before") or item.get("after"):
        before = [neighbor["summary"] for neighbor in item.get("before", [])]
        after = [neighbor["summary"] for neighbor in item.get("after", [])]
        text = _format_context_blocks(text, [("Before", before), ("After", after)])
    return text

In [ ]:
#| export
def filter_context(
    scope: str = ".",  # Project, folder, glob, or notebook used to choose notebooks
    query=None,  # Optional query string/dict/list using id/type/chapter/contains/regex/errors/export/header selectors
    include_re: str | None = None,  # Optional regex that matched cell source must include
    exclude_re: str | None = None,  # Optional regex that matched cell source must not include
    max_matches: int = 50,  # Maximum matching cells to return
    max_chars_per_cell: int = 1200,  # Maximum source characters shown per cell; <=0 means no cap
    view: str = "source",  # source, summary, or cell; cell implies stable line numbers
    line_numbers: bool = False,  # Include 1-based source line numbers in source views
    before: int = 0,  # Number of preceding neighbor cells to summarize for each match
    after: int = 0,  # Number of following neighbor cells to summarize for each match
    verbose: bool = True,  # Print rendered context; pass False to only return structured data
):
    "Show notebook cells in source order after applying query and regex filters."
    view = _filter_context_view(view)
    if query is None and include_re is None and exclude_re is None and view != "summary":
        raise ValueError("Pass query, include_re, exclude_re, or view='summary' to filter project context")
    matches, total = [], 0
    before, after = max(0, int(before or 0)), max(0, int(after or 0))
    for path in _context_notebooks(scope):
        nb = read_nb(path)
        for idx, cell in _query_items(nb, query):
            source = cell_source(cell)
            if not _context_match(source, include_re, exclude_re): continue
            total += 1
            if len(matches) >= max_matches: continue
            item = _filter_match_record(path, idx, cell, max_chars_per_cell)
            if before or after:
                neighbors = _neighbor_records(path, nb, idx, before=before, after=after)
                item["before"] = [n for n in neighbors if n["cell_idx"] < idx]
                item["after"] = [n for n in neighbors if n["cell_idx"] > idx]
            matches.append(item)
    nl = chr(10)
    filter_lines = [
        f"scope={scope}",
        f"query={query!r}",
        f"include_re={include_re!r}",
        f"exclude_re={exclude_re!r}",
        f"view={view!r}",
        f"before={before}",
        f"after={after}",
        f"max_matches={max_matches}",
    ]
    match_lines = [_render_filter_match(item, view=view, line_numbers=line_numbers) for item in matches]
    title = f"Filtered context: {scope}{nl}Matches: {len(matches)} shown of {total}"
    text = _format_context_blocks(title, [("Filters", filter_lines), ("Matches", match_lines)])
    return _context_result(
        "filter_context", text, verbose=verbose, scope=str(scope), query=query,
        include_re=include_re, exclude_re=exclude_re, max_matches=max_matches,
        max_chars_per_cell=max_chars_per_cell, view=view, line_numbers=line_numbers,
        before=before, after=after, total_matches=total, matches=matches,
    )

In [ ]:
def _write_read_sample_notebook(path):
    double_example = mk_cell("double(3)", cell_type="code")
    double_example.outputs = [{
        "output_type": "execute_result",
        "execution_count": 1,
        "metadata": {},
        "data": {"text/plain": "6"},
    }]
    add_example = mk_cell("add(2, 3)", cell_type="code")
    add_example.outputs = [{
        "output_type": "execute_result",
        "execution_count": 1,
        "metadata": {},
        "data": {"text/plain": "5"},
    }]
    calculator_example = mk_cell("Calculator(10).total(2)", cell_type="code")
    calculator_example.outputs = [{
        "output_type": "execute_result",
        "execution_count": 1,
        "metadata": {},
        "data": {"text/plain": "22"},
    }]
    error_cell = mk_cell("raise RuntimeError('boom')", cell_type="code")
    error_cell.outputs = [{
        "output_type": "error",
        "ename": "RuntimeError",
        "evalue": "boom",
        "traceback": ["RuntimeError: boom"],
    }]
    nb = new_nb([
        mk_cell("# Sample tool\nNotebook-level note.", cell_type="markdown"),
        mk_cell("More file-level context.", cell_type="markdown"),
        mk_cell("import math", cell_type="code"),
        mk_cell("## Math\nThis chapter demonstrates code-first groups.", cell_type="markdown"),
        mk_cell(
            "#| export\ndef double(value):\n"
            "    \"\"\"Double a value.\"\"\"\n"
            "    return value * 2",
            cell_type="code",
        ),
        mk_cell("Double a value by multiplying it by two.", cell_type="markdown"),
        double_example,
        mk_cell("assert double(3) == 6", cell_type="code"),
        mk_cell(
            "#| export\ndef add(a, b):\n"
            "    \"\"\"Add values.\"\"\"\n"
            "    return double(a) + b",
            cell_type="code",
        ),
        mk_cell("Add the doubled first value to the second.", cell_type="markdown"),
        add_example,
        mk_cell("assert add(1, 2) == 4", cell_type="code"),
        mk_cell(
            "#| export\nclass Calculator:\n"
            "    \"\"\"Calculate values.\"\"\"\n"
            "    def __init__(self, base):\n"
            "        \"\"\"Store the base value.\"\"\"\n"
            "        self.base = base\n\n"
            "    def total(self, value):\n"
            "        \"\"\"Add value to the base.\"\"\"\n"
            "        return add(self.base, value)",
            cell_type="code",
        ),
        mk_cell("Calculator keeps a base and adds a value.", cell_type="markdown"),
        calculator_example,
        mk_cell("assert Calculator(10).total(2) == 22", cell_type="code"),
        mk_cell("### Detail\nNested rationale mentioning add.", cell_type="markdown"),
        mk_cell("detail_value = add(2, 4)", cell_type="code"),
        error_cell,
    ])
    write_nb(nb, path)
    return nb

In [ ]:
with write_demo_notebook("01_read_filter_context_example.ipynb") as path:
    nb = new_nb([mk_cell("detail_value = 6"), mk_cell("other_value = 1")])
    write_nb(nb, path)
    filtered = filter_context(str(path), query="type=code regex=detail_value", verbose=False)
filtered["total_matches"], filtered["matches"][0]["cell_id"]

`filter_context` also doubles as a compact cell finder. Use `view="summary"` for one line per cell, `view="cell"` for a numbered single-cell view, flag queries such as `errors`, `export`, and `headers` for common notebook slices, `chapter=` for one header section, and `before`/`after` for grep-like neighbor summaries.

In [ ]:
with write_demo_notebook("01_read_cell_search_examples.ipynb") as path:
    nb = _write_read_sample_notebook(path)
    summary = filter_context(str(path), view="summary", verbose=False)
    numbered = filter_context(str(path), query=f"id={nb.cells[8].id}", view="cell", verbose=False)
    errors = filter_context(str(path), query="errors", verbose=False)
    exports = filter_context(str(path), query="export", view="summary", verbose=False)
    outline = filter_context(str(path), query="headers", view="summary", verbose=False)
    section = filter_context(str(path), query="chapter=Detail", verbose=False)
    grep = filter_context(str(path), query="regex=detail_value", view="summary", before=1, after=1, verbose=False)
summary["matches"][0]["summary"], numbered["text"].splitlines()[10], errors["matches"][0]["error"], len(exports["matches"]), len(outline["matches"]), section["total_matches"], grep["matches"][0]["before"][0]["cell_id"]

In [ ]:
#| hide
with write_demo_notebook("01_read_filter_context_check.ipynb") as path:
    nb = new_nb([mk_cell("detail_value = 6"), mk_cell("other_value = 1")])
    write_nb(nb, path)
    assert filter_context(str(path), include_re="detail_value", verbose=False)["total_matches"] == 1

In [ ]:
#| export
def _annotation_name(annotation):
    if annotation is None: return None
    if isinstance(annotation, ast.Name): return annotation.id
    if isinstance(annotation, ast.Attribute): return annotation.attr
    if isinstance(annotation, ast.Constant): return annotation.value
    return ast.unparse(annotation)

In [ ]:
#| export
def _first_arg_annotation(node):
    args = node.args.posonlyargs or node.args.args
    return _annotation_name(args[0].annotation) if args else None

In [ ]:
#| export
def _assignment_target_names(node):
    if isinstance(node, ast.Assign): raw_targets = node.targets
    elif isinstance(node, (ast.AnnAssign, ast.AugAssign)): raw_targets = [node.target]
    else: return []
    names = []
    for target in raw_targets:
        if isinstance(target, ast.Name): names.append(target.id)
        elif isinstance(target, (ast.Tuple, ast.List)):
            names.extend(elt.id for elt in target.elts if isinstance(elt, ast.Name))
    return names

In [ ]:
#| export
def _node_defines_symbol(node, symbol):
    parts = symbol.split(".")
    name = parts[-1]
    if len(parts) == 1 and name in _assignment_target_names(node): return True
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and node.name == name:
        return True
    if len(parts) < 2: return False
    cls_name, meth_name = parts[-2], parts[-1]
    if isinstance(node, ast.ClassDef) and node.name == cls_name:
        return any(isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) and child.name == meth_name for child in node.body)
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == meth_name:
        return _first_arg_annotation(node) == cls_name
    return False

In [ ]:
#| export
def _cell_defines_symbol(cell, symbol):
    if cell.cell_type != "code": return False
    try: tree = ast.parse(source_without_directives(cell.source))
    except SyntaxError: return False
    return any(_node_defines_symbol(node, symbol) for node in tree.body)

In [ ]:
#| export
def _find_symbol_cell(nb, symbol):
    for idx, cell in enumerate(nb.cells):
        if _cell_defines_symbol(cell, symbol): return idx
    raise ValueError(f"Could not find symbol {symbol!r}")

In [ ]:
#| export
def _find_symbol_node(cell, symbol):
    if getattr(cell, "cell_type", None) != "code": return None
    try: tree = ast.parse(source_without_directives(cell.source))
    except SyntaxError: return None
    parts = symbol.split(".")
    for node in tree.body:
        if len(parts) == 1 and symbol in _assignment_target_names(node): return node
        if len(parts) == 1 and isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and node.name == symbol:
            return node
        if _node_defines_symbol(node, symbol):
            if isinstance(node, ast.ClassDef) and len(parts) > 1:
                name = parts[-1]
                return next((child for child in node.body if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) and child.name == name), node)
            return node
    return None

In [ ]:
#| export
def _following_markdown(cells, idx, limit):
    docs = []
    pos = idx + 1
    while pos < len(cells) and len(docs) < limit:
        cell = cells[pos]
        if is_exported_code_cell(cell): break
        if getattr(cell, "cell_type", None) == "markdown": docs.append((pos, cell))
        pos += 1
    return docs

In [ ]:
#| export
def _following_examples(cells, idx, limit):
    examples = []
    pos = idx + 1
    while pos < len(cells) and len(examples) < limit:
        cell = cells[pos]
        if is_exported_code_cell(cell): break
        if getattr(cell, "cell_type", None) == "code": examples.append((pos, cell))
        pos += 1
    return examples

In [ ]:
#| export
def _symbol_signature_text(cell, symbol):
    node = _find_symbol_node(cell, symbol)
    if node is None: return _code_overview(cell)
    if isinstance(node, ast.ClassDef): return _class_overview(node)
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): return _function_overview(node)
    return [_source_for_node(cell, node)]

In [ ]:
#| export
def _usage_group_locations(raw_locations):
    raw = str(raw_locations or "").strip()
    if not raw or raw == "none": return [], 0
    grouped = {}
    for item in [part.strip() for part in raw.split(";") if part.strip()]:
        path, _, cell_id = item.rpartition(" id=")
        if not path: path, cell_id = item, ""
        grouped.setdefault(path, []).append(cell_id)
    return list(grouped.items()), sum(len(ids) for ids in grouped.values())

In [ ]:
#| export
def _format_usage_locations(label, raw_locations, max_paths=4, max_ids=4):
    groups, total = _usage_group_locations(raw_locations)
    if not total: return [f"{label}: none"]
    cell_word = "cell" if total == 1 else "cells"
    notebook_word = "notebook" if len(groups) == 1 else "notebooks"
    lines = [f"{label}: {total} {cell_word} across {len(groups)} {notebook_word}"]
    for path, ids in groups[:max_paths]:
        shown_ids = [item for item in ids[:max_ids] if item]
        suffix = f": {', '.join(shown_ids)}" if shown_ids else ""
        if len(ids) > max_ids: suffix += f", +{len(ids) - max_ids} more"
        lines.append(f"- {path}{suffix}")
    if len(groups) > max_paths: lines.append(f"- +{len(groups) - max_paths} more notebooks")
    return lines

In [ ]:
#| export
def _raw_caller_usage_lines(raw_lines):
    if "Caller usages:" not in raw_lines: return []
    start = raw_lines.index("Caller usages:") + 1
    return [line for line in raw_lines[start:] if line.startswith("- ")]

In [ ]:
#| export
def _format_symbol_usage(path, symbol):
    try:
        from nbskill.graph import symbol_usage_summary
        raw = symbol_usage_summary(path, [symbol])
    except Exception as exc:
        return [f"Usage unavailable: {type(exc).__name__}: {exc}"]
    if not raw: return []
    raw_lines = raw.splitlines()
    line = raw_lines[0]
    prefix = f"{symbol}: callers="
    if not line.startswith(prefix) or "; callees=" not in line: return raw_lines
    callers, _, callees = line[len(prefix):].partition("; callees=")
    lines = _format_usage_locations("Callers", callers)
    caller_usage = _raw_caller_usage_lines(raw_lines)
    if caller_usage:
        lines.append("Caller usages:")
        lines.extend(caller_usage)
    callee_items = [item.strip() for item in callees.split(";") if item.strip() and item.strip() != "none"]
    lines.append(f"Callees: {', '.join(callee_items)}" if callee_items else "Callees: none")
    return lines

In [ ]:
#| export
def symbol_context(
    path: str,  # Notebook path
    symbol: str,  # Function, class, or Class.method to inspect
    depth: int = 1,  # Callee summary depth; 0 keeps only direct implementation context
    verbose: bool = True,  # Print rendered context; pass False to only return structured data
):
    "Show a notebook symbol as implementation, trailing Docs, examples/tests, callers, and callees."
    nb = read_nb(path)
    idx, cell, node = _symbol_location(path, nb, symbol)
    source = _source_for_node(cell, node) if node is not None else cell_source(cell).strip()
    markdown = [
        {"cell_id": getattr(item, "id", ""), "cell_idx": pos, "source": cell_source(item).strip()}
        for pos, item in _following_markdown(nb.cells, idx, len(nb.cells))
    ]
    examples = [
        {
            **item,
            "source": _trim_context_source(item.get("source", ""), _CONTEXT_CELL_SOURCE_CHARS),
            "output": _trim_context_source(item.get("output", ""), _CONTEXT_CELL_OUTPUT_CHARS),
        }
        for item in _example_test_records(nb, symbol, definition_idx=idx)
    ]
    callers, callees = [], []
    if depth > 0:
        try:
            from nbskill.graph import symbol_graph_data
            graph_data = symbol_graph_data(path, symbol)
            callers = graph_data.get("caller_usages", [])
        except Exception:
            graph_data = None
            callers = []
        callees = _callee_summary(path, symbol, depth, graph_data=graph_data)
    nl = chr(10)
    blocks = [
        ("Implementation", source),
        ("Docs", [f"Cell id={item['cell_id']}{nl}{item['source']}" for item in markdown]),
        ("Examples/tests", [
            f"Cell id={item['cell_id']} {item['kind']}{nl}{item['source']}" + (f"{nl}Output:{nl}{item['output']}" if item.get("output") else "")
            for item in examples
        ]),
        ("Callers", [
            f"- {item.get('path')} id={item.get('cell_id')} line {item.get('lineno')}: {item.get('line')}"
            for item in callers
        ]),
        (f"Callees depth {depth}", _format_callee_items(callees)),
    ]
    text = _format_context_blocks(f"Symbol context: {symbol}{nl}Location: {path} {cell_prefix(idx, cell, True)}", blocks)
    return _context_result(
        "symbol_context", text, verbose=verbose, path=str(path), symbol=symbol,
        depth=depth, location={"cell_id": getattr(cell, "id", ""), "cell_idx": idx},
        source=source, markdown=markdown, examples=examples, callers=callers, callees=callees, symbols=[symbol],
    )

In [ ]:
#| hide
with write_tool_notebook("01_read_tool.ipynb") as path:
    file_view = file_context(str(path), verbose=False)
    test_eq(file_view["definitions"][0]["symbol"], "double_answer")
    chapter_view = chapter_context(str(path), name="Arithmetic", verbose=False)
    test_eq(chapter_view["chapter"]["title"], "Arithmetic")
    symbol_view = symbol_context(str(path), "double_answer", depth=0, verbose=False)
    test_eq(symbol_view["location"]["cell_id"], "tool-double-answer")

### Reading through the model

`Notebook.context` and `Notebook.symbol` let callers read a notebook through the model instead of threading paths around, mirroring `file_context` and `symbol_context`.

In [ ]:
#| export
@patch
def context(self: Notebook, **kw):
    'Show this notebook file context (see `file_context`).'
    return file_context(self.path, **kw)

In [ ]:
#| export
@patch
def symbol(self: Notebook, symbol, **kw):
    'Show implementation context for `symbol` (see `symbol_context`).'
    return symbol_context(self.path, symbol, **kw)

In [ ]:
#| hide
from nbskill.foundation import Notebook, write_demo_notebook

In [ ]:
#| hide
with write_demo_notebook('notebook_context_demo.ipynb') as demo:
    nb = Notebook.from_path(demo)
    assert isinstance(nb.context(verbose=False), dict)
    assert isinstance(nb.symbol('demo_answer', verbose=False), dict)

In [ ]:
#| hide
with write_demo_notebook("01_read_doc.ipynb") as path:
    nb = new_nb([
        mk_cell("# Demo project\nFile-level note.", cell_type="markdown"),
        mk_cell("## Addition", cell_type="markdown"),
        mk_cell("#| export\ndef add(a, b):\n    \"\"\"Add two values.\"\"\"\n    return a + b", cell_type="code"),
        mk_cell("This explains add.", cell_type="markdown"),
        mk_cell("add(2, 3)", cell_type="code"),
        mk_cell("assert add(1, 2) == 3", cell_type="code"),
    ])
    write_nb(nb, path)
    result = symbol_context(str(path), "add", depth=0, verbose=False)
    assert "Add two values." in result["text"]
    assert "This explains add." in result["text"]
    assert "add(2, 3)" in result["text"]
    assert "assert add(1, 2) == 3" in result["text"]
    assert result["text"].index("Add two values.") < result["text"].index("This explains add.") < result["text"].index("add(2, 3)")

In [ ]:
#| export
def _context_cell_semantic_type(cell):
    return NotebookCell(cell).semantic_type()

In [ ]:
#| export
def _context_cell_record(idx, cell):
    record = NotebookCell(cell, idx=idx).context_record()
    record["source"] = _trim_context_source(record.get("source", ""), _CONTEXT_CELL_SOURCE_CHARS)
    record["output"] = _trim_context_source(record.get("output", ""), _CONTEXT_CELL_OUTPUT_CHARS)
    return record

In [ ]:
#| export
def _render_context_cell(item):
    nl = chr(10)
    header = f"Cell id={item['cell_id']} idx={item['cell_idx']} [{item['cell_type']}] ({item['semantic_type']})"
    text = f"{header}{nl}{item['source']}".rstrip()
    if item.get("output"): text = f"{text}{nl}Output:{nl}{item['output']}"
    return text

In [ ]:
#| export
def _exported_definition_records(path, nb):
    return [item for item in _definition_records(path, nb) if is_exported_code_cell(nb.cells[item["cell_idx"]])]

In [ ]:
#| export
def _definition_records_for_cell(path, nb, cell_idx):
    return [item for item in _definition_records(path, nb) if item.get("cell_idx") == cell_idx]

In [ ]:
#| export
def _symbol_source(path, nb, symbol):
    idx, cell, node = _symbol_location(path, nb, symbol)
    source = _source_for_node(cell, node) if node is not None else cell_source(cell).strip()
    return idx, cell, node, source

In [ ]:
#| export
def _source_mentions_symbols(source, symbols):
    for symbol in symbols:
        short = symbol_short_name(symbol)
        if symbol in source or re.search(rf"\b{re.escape(short)}\b", source): return True
    return False

In [ ]:
#| export
def _related_cell_records(nb, symbols, definition_cell_idx=None):
    records = []
    for idx, cell in enumerate(nb.cells):
        if idx == definition_cell_idx: continue
        semantic = _context_cell_semantic_type(cell)
        if semantic not in {"markdown", "example", "test"}: continue
        if _source_mentions_symbols(cell_source(cell), symbols): records.append(_context_cell_record(idx, cell))
    return records

In [ ]:
#| export
def _single_definition_for_cell(path, nb, cell_idx):
    definitions = _definition_records_for_cell(path, nb, cell_idx)
    if len(definitions) > 1:
        names = ", ".join(item["symbol"] for item in definitions)
        raise ValueError(f"Cell id={getattr(nb.cells[cell_idx], 'id', '')} contains multiple definitions: {names}")
    return definitions[0] if definitions else None

In [ ]:
#| export
def _implementation_context(path, idx, cell, symbol, source, verbose=True):
    nl = chr(10)
    text = _format_context_blocks(
        f"Implementation context: {symbol}{nl}Location: {path} {cell_prefix(idx, cell, True)}",
        [("Implementation", source)],
    )
    return _context_result(
        "implementation_context", text, verbose=verbose, path=str(path), symbol=symbol,
        location={"cell_id": getattr(cell, "id", ""), "cell_idx": idx}, source=source, symbols=[symbol],
    )

In [ ]:
#| export
def _mentions_context(path, nb, symbols, definition_cell_idx=None, verbose=True):
    records = _related_cell_records(nb, symbols, definition_cell_idx=definition_cell_idx)
    label = ", ".join(symbols)
    body = [_render_context_cell(item) for item in records] or [f"No markdown, example, or test cells mention {label}."]
    nl = chr(10)
    text = _format_context_blocks(f"Related context: {label}{nl}Path: {path}", [("Related cells", body)])
    return _context_result(
        "mentions_context", text, verbose=verbose, path=str(path), symbols=symbols,
        definition_cell_idx=definition_cell_idx, mentions=records,
    )

In [ ]:
#| export
def _cell_context(path, cell_idx, overview=False, verbose=True):
    nb = read_nb(path)
    cell = nb.cells[cell_idx]
    definition = _single_definition_for_cell(path, nb, cell_idx)
    if definition is None:
        item = _context_cell_record(cell_idx, cell)
        text = _format_context_blocks(f"Cell context: {path} {cell_prefix(cell_idx, cell, True)}", [("Cell", _render_context_cell(item))])
        return _context_result("cell_context", text, verbose=verbose, path=str(path), cell=item, symbols=[])
    symbol = definition["symbol"]
    idx, cell, node, source = _symbol_source(path, nb, symbol)
    if overview: return _implementation_context(path, idx, cell, symbol, source, verbose=verbose)
    return symbol_context(path, symbol, depth=1, verbose=verbose)

In [ ]:
#| export
def _symbol_focus_context(path, symbol, overview=False, verbose=True):
    nb = read_nb(path)
    idx, cell, node, source = _symbol_source(path, nb, symbol)
    definition_node = isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))
    if overview or not definition_node: return _implementation_context(path, idx, cell, symbol, source, verbose=verbose)
    return symbol_context(path, symbol, depth=1, verbose=verbose)

In [ ]:
#| export
def _markdown_heading_level(source):
    for line in str(source or "").splitlines():
        match = re.match(r"^(#{1,6})\s+", line.strip())
        if match: return len(match.group(1))
    return None

In [ ]:
#| export
def _notebook_markdown_records(cells):
    return [
        {
            "cell_id": getattr(cell, "id", ""),
            "cell_idx": idx,
            "source": cell_source(cell).strip(),
            "heading_level": _markdown_heading_level(cell_source(cell)),
        }
        for idx, cell in enumerate(cells)
        if getattr(cell, "cell_type", None) == "markdown" and cell_source(cell).strip()
    ]

In [ ]:
#| export
def _notebook_overview_markdown_records(cells):
    records = _notebook_markdown_records(cells)
    first_chapter = next((item["cell_idx"] for item in records if item["heading_level"] == 2), None)
    if first_chapter is None: return records
    wanted = {idx for idx, _ in _notebook_head_items(cells)}
    for span in _chapter_spans_for_nb(cells): wanted.update(idx for idx, _ in _chapter_intro_markdown_items(cells, span))
    return [item for item in records if item["cell_idx"] in wanted]

In [ ]:
#| export
def _render_markdown_record(item):
    return f"Cell id={item['cell_id']}\n{item['source']}"

In [ ]:
#| export
def _render_definition_record(item):
    nl = chr(10)
    return f"Cell id={item['cell_id']} {item['kind']} {item['symbol']}{nl}{item['text']}"

In [ ]:
#| export
def _notebook_context(
    path: str,
    scope: str = ".",
    overview: bool = False,
    verbose: bool = True,
    readme_sections: int = 2,
):
    "Show one notebook; overview gives headings and counts, while default output includes every cell."
    nb = read_nb(path)
    root = _context_root(scope if scope not in (None, "") else path)
    definitions = _definition_records(path, nb)
    exported_definitions = _exported_definition_records(path, nb)
    markdown = _notebook_markdown_records(nb.cells)
    cells = [_context_cell_record(idx, cell) for idx, cell in enumerate(nb.cells)]
    nl = chr(10)
    title = f"Notebook context: {Path(path).name}{nl}Path: {path}{nl}Project: {root}"
    if overview:
        overview_markdown = _notebook_overview_markdown_records(nb.cells)
        blocks = [
            ("Markdown overview", [_render_markdown_record(item) for item in overview_markdown]),
            ("Contents", [f"cells={len(cells)}", f"definitions={len(definitions)}", f"exported_definitions={len(exported_definitions)}"]),
        ]
        text = _format_context_blocks(title, blocks)
        return _context_result(
            "notebook_context", text, verbose=verbose, path=str(path), scope=str(scope), root=str(root),
            overview=True, cell_count=len(cells), definition_count=len(definitions),
            exported_definition_count=len(exported_definitions), cells=[], readme_sections=[], imports=[],
            markdown=overview_markdown, next_action="Pass an explicit symbol target or resolution='full' for definitions.",
        )
    blocks = [
        ("Full cells", [_render_context_cell(item) for item in cells]),
        ("Function signatures", [_render_definition_record(item) for item in definitions]),
    ]
    text = _format_context_blocks(title, blocks)
    return _context_result(
        "notebook_context", text, verbose=verbose, path=str(path), scope=str(scope), root=str(root),
        overview=False, definitions=definitions, cells=cells, readme_sections=[], imports=[], markdown=markdown,
    )

In [ ]:
#| export
def _context_cell_matches(target, notebooks):
    matches = []
    for path in notebooks:
        nb = read_nb(path)
        for idx, cell in enumerate(nb.cells):
            if getattr(cell, "id", None) == target:
                matches.append(dict(path=str(path), cell_idx=idx, cell_id=target))
    return matches

In [ ]:
#| export
def _context_symbol_matches(target, notebooks):
    matches = []
    for path in notebooks:
        nb = read_nb(path)
        try: idx = _find_symbol_cell(nb, target)
        except ValueError: continue
        if idx is not None:
            matches.append(dict(path=str(path), cell_idx=idx, symbol=str(target)))
    return matches

In [ ]:
#| export
def _context_chapter_matches(target, notebooks):
    chapter_matches, subchapter_matches = [], []
    for path in notebooks:
        nb = read_nb(path)
        for span in _chapter_spans_for_nb(nb.cells):
            if _chapter_matches(span, target):
                chapter_matches.append(dict(path=str(path), chapter=span))
        for span in _all_heading_spans_for_nb(nb.cells):
            if _chapter_matches(span, target):
                subchapter_matches.append(dict(path=str(path), chapter=span))
    return chapter_matches or subchapter_matches

In [ ]:
#| export
def _context_candidate_target(item):
    path = item.get("path", "")
    detail = item.get("cell_id") or item.get("symbol") or item.get("chapter", {}).get("title", "")
    return f"{path}#{detail}" if path and detail else (path or detail)

In [ ]:
#| export
def _context_candidate_line(kind, item):
    detail = item.get("cell_id") or item.get("symbol") or item.get("chapter", {}).get("title", "")
    target = _context_candidate_target(item)
    suffix = f" use {target!r}" if target else ""
    return f"- {kind}: {item.get('path', '')} {detail}{suffix}".rstrip()

In [ ]:
#| export
def _context_one_match(kind, target, matches):
    if len(matches) == 1: return matches[0]
    if not matches: return None
    candidates = [_context_candidate_target(item) for item in matches[:8]]
    shown = [_context_candidate_line(kind, item) for item in matches[:8]]
    message = (
        f"Target {target!r} matched multiple {kind} contexts. "
        "Use a qualified target like path.ipynb#cell_id or path.ipynb#symbol.\n"
        + "\n".join(shown)
    )
    exc = ValueError(message)
    exc.candidates = [item for item in candidates if item]
    raise exc

In [ ]:
#| export
def _context_search_root(scope):
    raw = "." if scope in (None, "") else str(scope)
    path = Path(raw).expanduser()
    candidates = [path] if path.is_absolute() else [path, Path.cwd() / path]
    for candidate in candidates:
        if candidate.exists(): return candidate.parent.resolve() if candidate.is_file() else candidate.resolve()
    if any(char in raw for char in "*?[]"):
        parent = path.parent if str(path.parent) else Path(".")
        if not parent.is_absolute(): parent = Path.cwd() / parent
        if parent.exists(): return parent.resolve()
    return _context_root(raw)

In [ ]:
#| export
def _context_display_path(path, root):
    try: return Path(path).resolve().relative_to(Path(root).resolve()).as_posix()
    except (OSError, ValueError): return str(path)

In [ ]:
#| export
def _context_file_paths(root):
    root = Path(root)
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [name for name in dirnames if name not in _CONTEXT_SEARCH_SKIP_DIRS]
        for filename in filenames:
            path = Path(dirpath) / filename
            if path.suffix.lower() in _CONTEXT_SEARCH_SKIP_SUFFIXES: continue
            yield path

In [ ]:
#| export
def _context_line_snippet(lines, idx, radius=1):
    start = max(0, idx - radius)
    stop = min(len(lines), idx + radius + 1)
    return "\n".join(f"{pos + 1} | {lines[pos]}" for pos in range(start, stop))

In [ ]:
#| export
def _context_plain_file_matches(target, root, max_matches):
    matches, total = [], 0
    for file_count, path in enumerate(_context_file_paths(root), start=1):
        if file_count > _CONTEXT_SEARCH_MAX_FILES: break
        try:
            if path.stat().st_size > _CONTEXT_SEARCH_MAX_BYTES: continue
            text = path.read_text(encoding="utf-8", errors="ignore")
        except OSError: continue
        if target not in text: continue
        lines = text.splitlines() or [""]
        for idx, line in enumerate(lines):
            if target not in line: continue
            total += 1
            if len(matches) >= max_matches: continue
            matches.append(dict(path=_context_display_path(path, root), line=idx + 1, snippet=_context_line_snippet(lines, idx)))
    return matches, total

In [ ]:
#| export
def _context_notebook_text_matches(target, notebooks, max_matches):
    matches, total = [], 0
    for path in notebooks:
        nb = read_nb(path)
        for idx, cell in enumerate(nb.cells):
            source = cell_source(cell)
            if target not in source: continue
            total += 1
            if len(matches) >= max_matches: continue
            matches.append(dict(path=str(path), cell_id=getattr(cell, "id", ""), cell_idx=idx, cell_type=getattr(cell, "cell_type", ""), source=_trim_context_source(source, _CONTEXT_CELL_SOURCE_CHARS)))
    return matches, total

In [ ]:
#| export
def _literal_search_context(target, scope, notebooks, max_matches=50, max_chars_per_match=1200, verbose=True):
    root = _context_search_root(scope)
    notebook_matches, notebook_total = _context_notebook_text_matches(target, notebooks, max_matches)
    file_budget = max(0, max_matches - len(notebook_matches))
    file_matches, file_total = _context_plain_file_matches(target, root, file_budget)
    total, nl = notebook_total + file_total, chr(10)
    notebook_lines, file_lines = [], []
    for item in notebook_matches:
        header = f"{item['path']} id={item['cell_id']} idx={item['cell_idx']} type={item['cell_type']}{nl}"
        notebook_lines.append(header + _trim_context_source(item["source"], max_chars_per_match))
    for item in file_matches: file_lines.append(f"{item['path']}:{item['line']}{nl}{_trim_context_source(item['snippet'], max_chars_per_match)}")
    shown = len(notebook_matches) + len(file_matches)
    title = f"Search context: {target}{nl}Scope: {scope}{nl}Matches: {shown} shown of {total}"
    text = _format_context_blocks(title, [("Notebook cells", notebook_lines), ("Files", file_lines)])
    if not total: text = f"{title}{nl}{nl}No text matches found."
    data = dict(root=str(root), total_matches=total, notebook_matches=notebook_matches, file_matches=file_matches)
    return _context_result("search_context", text, verbose=verbose, target=str(target), scope=str(scope), **data)

In [ ]:
#| export
_CONTEXT_MODES = {"auto", "edit", "review", "overview"}

In [ ]:
#| export
def _context_mode(mode):
    mode = "auto" if mode in (None, "") else str(mode).lower()
    if mode not in _CONTEXT_MODES:
        raise ValueError(f"mode must be one of {sorted(_CONTEXT_MODES)}, not {mode!r}")
    return mode

In [ ]:
#| export
def _context_around(around):
    try: return max(0, int(around or 0))
    except (TypeError, ValueError) as exc:
        raise ValueError("around must be an integer") from exc

In [ ]:
#| export
def _context_target_items(targets):
    if targets is None: return None
    if isinstance(targets, str):
        return [targets] if targets else []
    return [str(item) for item in targets if str(item)]

In [ ]:
#| export
def _context_symbol_graphs(path, symbols):
    symbols = [str(item) for item in dict.fromkeys(symbols or []) if item]
    if not symbols: return []
    try:
        from nbskill.graph import symbol_graph_data, symbol_graph_public_data
    except Exception:
        return []
    graphs = []
    for symbol in symbols:
        try: graphs.append(symbol_graph_public_data(symbol_graph_data(path, symbol)))
        except Exception: pass
    return graphs

In [ ]:
#| export
def _context_graph_text(graphs):
    if not graphs: return ""
    lines = ["Symbol graph"]
    for graph in graphs:
        symbol = graph.get("symbol", "")
        callers = graph.get("caller_usages") or graph.get("callers") or []
        callees = graph.get("callees") or []
        lines.append(f"- {symbol}: definitions={len(graph.get('definitions', []))}, callers={len(callers)}, callees={len(callees)}")
        for usage in callers[:5]:
            line = usage.get("line", "")
            loc = f"{usage.get('path')} id={usage.get('cell_id')}"
            lineno = f" line {usage.get('lineno')}" if usage.get("lineno") else ""
            lines.append(f"  caller: {loc}{lineno}: {line}".rstrip())
        for callee in callees[:5]:
            callee_symbol = callee.get("symbol", callee) if isinstance(callee, dict) else callee
            lines.append(f"  callee: {callee_symbol}")
    return "\n".join(lines)

In [ ]:
#| export
def _context_cell_symbols(path, cell_idx):
    nb = read_nb(path)
    return [item["symbol"] for item in _definition_records_for_cell(path, nb, cell_idx)]

In [ ]:
#| export
def _context_with_graphs(result, graphs):
    if not graphs: return result
    text = result.get("text", "")
    graph_text = _context_graph_text(graphs)
    return {**result, "text": f"{text}\n\n{graph_text}".rstrip(), "symbol_graphs": graphs}

In [ ]:
#| export
def _context_neighbor_records(path, cell_idx, around):
    around = _context_around(around)
    if around <= 0: return []
    nb = read_nb(path)
    start, end = max(0, cell_idx - around), min(len(nb.cells), cell_idx + around + 1)
    return [{**_context_cell_record(idx, nb.cells[idx]), "path": str(path)} for idx in range(start, end) if idx != cell_idx]

In [ ]:
#| export
def _context_append_block(text, heading, items):
    items = [item for item in items if item]
    if not items: return text
    return f"{text.rstrip()}\n\n{heading}\n" + "\n\n".join(items)

In [ ]:
#| export
def _context_with_neighbors(result, path, cell_idx, around):
    neighbors = _context_neighbor_records(path, cell_idx, around)
    if not neighbors: return result
    text = _context_append_block(result.get("text", ""), "Neighbor cells", [_render_context_cell(item) for item in neighbors])
    return {**result, "text": text, "neighbors": neighbors}

In [ ]:
#| export
def _context_edit_result(path, cell_idx, symbol=None, source=None, verbose=True):
    nb = read_nb(path)
    cell = nb.cells[cell_idx]
    source = cell_source(cell) if source is None else source
    cell_record = {**_context_cell_record(cell_idx, cell), "path": str(path)}
    symbols = [symbol] if symbol else _context_cell_symbols(path, cell_idx)
    related = _symbol_focus_context(path, symbol, verbose=False) if symbol else {}
    docs, examples = related.get("markdown", []), related.get("examples", [])
    callers, callees = related.get("callers", []), related.get("callees", [])
    title = f"Edit context: {symbol or cell_record['cell_id']}\nPath: {path}\nCell: {cell_prefix(cell_idx, cell, True)}"
    nl = chr(10)
    example_text = [f"{item['kind']}{nl}{item['source']}" + (f"{nl}Output:{nl}{item['output']}" if item.get("output") else "") for item in examples]
    caller_text = [f"- {item.get('path')} id={item.get('cell_id')} line {item.get('lineno')}: {item.get('line')}" for item in callers]
    blocks = [("Source", source), ("Docs", [item["source"] for item in docs]), ("Examples/tests", example_text), ("Callers", caller_text), ("Callees", _format_callee_items(callees))]
    text = _format_context_blocks(title, blocks)
    result = dict(verbose=verbose, path=str(path), symbol=symbol, cell=cell_record)
    result.update(cell_id=cell_record["cell_id"], cell_idx=cell_idx, source=source, expected_hash=source_hash(cell_source(cell)))
    result.update(symbols=symbols, docs=docs, examples=examples, callers=callers, callees=callees)
    return _context_result("edit_context", text, **result)

In [ ]:
#| export
def _context_cell_payload(path, cell_idx, mode, around):
    if mode == "edit":
        result, graphs = _context_edit_result(path, cell_idx, verbose=False), []
    else:
        result = _cell_context(path, cell_idx, overview=(mode == "overview"), verbose=False)
        graphs = [] if mode == "overview" else _context_symbol_graphs(path, result.get("symbols", []))
        result = _context_with_graphs(result, graphs)
    return _context_with_neighbors(result, path, cell_idx, around), graphs

In [ ]:
#| export
def _context_symbol_payload(path, symbol, mode, around):
    nb = read_nb(path)
    idx, cell, node, source = _symbol_source(path, nb, symbol)
    if mode == "edit":
        result, graphs = _context_edit_result(path, idx, symbol=symbol, source=source, verbose=False), []
    else:
        result = _symbol_focus_context(path, symbol, overview=(mode == "overview"), verbose=False)
        graphs = [] if mode == "overview" else _context_symbol_graphs(path, result.get("symbols", [symbol]))
        result = _context_with_graphs(result, graphs)
    return _context_with_neighbors(result, path, idx, around), graphs

In [ ]:
#| export
def _context_as_single(result, target, scope, resolved_kind, mode="auto", around=0, **extra):
    return _context_result(
        "context", result["text"], verbose=True, ok=True, target=str(target), scope=str(scope),
        mode=mode, around=around, resolved_kind=resolved_kind, selection=result, **extra,
    )

In [ ]:
#| export
def _context_parse_qualified_target(target):
    text = str(target or "")
    if "#" not in text: return None
    path_text, ref = text.rsplit("#", 1)
    if not ref or ".ipynb" not in path_text: return None
    path = _context_existing_notebook(path_text)
    return (str(path), ref) if path is not None else None

In [ ]:
#| export
def _context_cell_index_for_ref(nb, ref):
    if str(ref).isdigit():
        idx = int(ref)
        if 0 <= idx < len(nb.cells): return idx
    for idx, cell in enumerate(nb.cells):
        if getattr(cell, "id", None) == ref: return idx
    return None

In [ ]:
#| export
def _context_qualified_candidates(path, nb, limit=8):
    cells = [f"{path}#{getattr(cell, 'id', '')}" for cell in nb.cells if getattr(cell, "id", "")]
    symbols = [f"{path}#{item['symbol']}" for item in _definition_records(path, nb)]
    return (cells[: limit // 2] + symbols)[:limit]

In [ ]:
#| export
def _context_unknown_qualified(path, ref, nb):
    candidates = _context_qualified_candidates(path, nb)
    shown = "\n".join(f"- {item}" for item in candidates)
    suffix = f"\nCandidates:\n{shown}" if shown else ""
    exc = ValueError(f"No cell id, cell index, or symbol {ref!r} in {path}.{suffix}")
    exc.candidates = candidates
    raise exc

In [ ]:
#| export
def _context_qualified_payload(target, mode, around):
    parsed = _context_parse_qualified_target(target)
    if parsed is None: return None
    path, ref = parsed
    nb = read_nb(path)
    cell_idx = _context_cell_index_for_ref(nb, ref)
    if cell_idx is not None:
        result, graphs = _context_cell_payload(path, cell_idx, mode, around)
        return result, "cell", {"symbol_graphs": graphs} if graphs else {}
    try:
        result, graphs = _context_symbol_payload(path, ref, mode, around)
        return result, "symbol", {"symbol_graphs": graphs} if graphs else {}
    except ValueError:
        _context_unknown_qualified(path, ref, nb)

In [ ]:
#| export
def _context_single(target="project", scope=".", mode="auto", around=0, verbose=True):
    target = "project" if target in (None, "") else str(target)
    scope = "." if scope in (None, "") else str(scope)
    mode, around = _context_mode(mode), _context_around(around)
    if target == "project":
        result = project_context(scope, verbose=False)
        return _context_as_single(result, target, scope, "project", mode=mode, around=around)
    python_qualified = _context_python_qualified_payload(target)
    if python_qualified is not None:
        result, resolved_kind, extra = python_qualified
        return _context_as_single(result, target, scope, resolved_kind, mode=mode, around=around, **extra)
    qualified = _context_qualified_payload(target, mode, around)
    if qualified is not None:
        result, resolved_kind, extra = qualified
        return _context_as_single(result, target, scope, resolved_kind, mode=mode, around=around, **extra)
    python_scope = _context_python_scope_payload(target, scope)
    if python_scope is not None:
        result, resolved_kind, extra = python_scope
        return _context_as_single(result, target, scope, resolved_kind, mode=mode, around=around, **extra)
    py_path = _context_existing_python(target)
    if py_path is not None:
        result = python_file_context(str(py_path), verbose=False)
        return _context_as_single(result, target, scope, "python_file", mode=mode, around=around)
    path = _context_existing_notebook(target)
    try: notebooks = _context_notebooks(scope)
    except ValueError: notebooks = []
    path = path or _context_named_notebook(target, notebooks)
    if path is not None:
        result = _notebook_context(path, scope=scope, overview=(mode in {"overview", "edit"}), verbose=False)
        return _context_as_single(result, target, scope, "notebook", mode=mode, around=around)
    cell = _context_one_match("cell", target, _context_cell_matches(target, notebooks))
    if cell is not None:
        result, graphs = _context_cell_payload(cell["path"], cell["cell_idx"], mode, around)
        return _context_as_single(result, target, scope, "cell", mode=mode, around=around, symbol_graphs=graphs)
    symbol = _context_one_match("symbol", target, _context_symbol_matches(target, notebooks))
    if symbol is not None:
        result, graphs = _context_symbol_payload(symbol["path"], target, mode, around)
        return _context_as_single(result, target, scope, "symbol", mode=mode, around=around, symbol_graphs=graphs)
    chapter = _context_one_match("chapter", target, _context_chapter_matches(target, notebooks))
    if chapter is not None:
        result = chapter_context(chapter["path"], name=chapter["chapter"]["title"], overview=(mode == "overview"), verbose=False)
        return _context_as_single(result, target, scope, "chapter", mode=mode, around=around)
    result = _literal_search_context(target, scope, notebooks, verbose=False)
    return _context_as_single(result, target, scope, "search", mode=mode, around=around)

In [ ]:
#| export
def _context_error_entry(target, exc):
    return {
        "kind": "context_error", "ok": False, "target": str(target),
        "error": {"type": type(exc).__name__, "message": str(exc)},
        "candidates": list(getattr(exc, "candidates", []) or []),
        "text": f"Context error: {target}\n{type(exc).__name__}: {exc}",
    }

In [ ]:
#| export
def _context_batch(targets, scope=".", mode="auto", around=0):
    scope = "." if scope in (None, "") else str(scope)
    items = _context_target_items(targets) or []
    results, rendered = [], []
    for item in items:
        try:
            result = _context_single(item, scope=scope, mode=mode, around=around, verbose=False)
            result = {**result, "ok": True}
        except Exception as exc:
            result = _context_error_entry(item, exc)
        results.append(result)
        rendered.append(result.get("text", ""))
    text = _format_context_blocks(f"Context batch\nTargets: {len(items)}", [("Results", "\n\n".join(rendered))])
    return _context_result(
        "context_batch", text, verbose=True, ok=all(item.get("ok") for item in results),
        targets=items, scope=scope, mode=mode, around=around, results=results,
    )

In [ ]:
#| export
def context(
    target: str = "project",
    scope: str = ".",
    targets: list[str] | None = None,
    mode: str = "auto",
    around: int = 0,
):
    "Return notebook-aware context; edit includes bounded docs, usages, and direct impact."
    mode = _context_mode(mode)
    around = _context_around(around)
    if targets is not None: return _context_batch(targets, scope=scope, mode=mode, around=around)
    return _context_single(target=target, scope=scope, mode=mode, around=around)

### Context modes and edit handoff

Use `mode="auto"`, `"overview"`, `"edit"`, or `"review"`; use `resolution` to choose summary, implementation, or full depth. Overview returns notebook identity, headings, counts, and a next action.

`mode="edit"` is the bounded handoff for a named symbol: the exact source and `expected_hash`, followed by its trailing Docs, local examples/tests, and direct caller/callee metadata. These sections are tied to the resolved symbol, so an agent can prepare an edit without widening the search. Use `around` only when physical neighbor cells are also needed.

In [ ]:
with write_demo_notebook(
    "01_read_context_example.ipynb",
    cells=[
        mk_cell("#| export\ndef add(a, b):\n    return a + b"),
        mk_cell("Add two values.", cell_type="markdown"),
        mk_cell("add(2, 3)"),
    ],
) as path:
    result = context(f"{path}#add", scope=str(path), mode="edit")
    selection = result["selection"]
    print(selection["cell_id"], selection["expected_hash"])
    print(selection["docs"][0]["source"], selection["examples"][0]["source"])

#### A compact context result

The public reader returns structured data and can also render the first useful line for a human.

In [ ]:
with write_demo_notebook("01_read_output.ipynb") as path:
    result = context("project", scope=str(path))
    print(result["kind"])
    print(result["text"].splitlines()[0])

In [ ]:
#| hide
with write_demo_notebook("01_read_output_test.ipynb") as path:
    result = context("project", scope=str(path))
    assert result["kind"] == "context"

`context` keeps the common call useful for implementation work. For example, `context("add", scope="nbs/01_read.ipynb")` resolves the symbol and prints the implementation, trailing Docs, examples/tests, callers, and callees; `mode="overview"` keeps only the implementation source. A notebook target such as `context("nbs/01_read.ipynb")` prints every cell in source order, while `mode="overview"` keeps the main notebook description, chapter headings, and function signatures.

In [ ]:
#| eval: false
context("add", scope="nbs/01_read.ipynb")

In [ ]:
#| hide
def _captured_context(*args, **kwargs):
    out = StringIO()
    with redirect_stdout(out):
        return context(*args, **kwargs)

with write_demo_notebook("01_read_single_context.ipynb") as path:
    nb = _write_read_sample_notebook(path)
    symbol = _captured_context("add", scope=str(path))
    symbol_overview = _captured_context("add", scope=str(path), mode="overview")
    notebook = _captured_context(str(path))
    notebook_overview = _captured_context(str(path), mode="overview")
    chapter = _captured_context("Math", scope=str(path))
    chapter_overview = _captured_context("Math", scope=str(path), mode="overview")
    cell = _captured_context(nb.cells[8].id, scope=str(path))
    cell_overview = _captured_context(nb.cells[8].id, scope=str(path), mode="overview")
    project = _captured_context("project", scope=str(path))
    with write_demo_notebook("01_read_context_files/search.txt") as plain_path:
        plain_path.write_text("plain file install_nbdev_hooks marker\n", encoding="utf-8")
        file_search = _captured_context("install_nbdev_hooks", scope=str(plain_path))
    cell_search = _captured_context("Nested rationale mentioning add", scope=str(path))
assert symbol["kind"] == "context"
assert symbol["resolved_kind"] == "symbol"
assert "Implementation" in symbol["text"]
assert "Examples/tests" in symbol["text"]
assert "Add the doubled first value to the second." in symbol["text"]
assert "def add(a, b):" in symbol["text"]
assert "double" in symbol["text"]
assert symbol_overview["selection"]["kind"] == "implementation_context"
assert "def add(a, b):" in symbol_overview["text"]
assert notebook["resolved_kind"] == "notebook"
assert "Notebook context:" in notebook["text"]
assert "Full cells" in notebook["text"]
assert "Function signatures" in notebook["text"]
assert "Nested rationale" in notebook["text"]
assert "assert add(1, 2) == 4" in notebook["text"]
assert "detail_value = add(2, 4)" in notebook["text"]
assert len(notebook["selection"]["cells"]) == len(nb.cells)
assert notebook_overview["selection"]["overview"] is True
assert "Markdown overview" in notebook_overview["text"]
assert "Contents" in notebook_overview["text"]
assert "Notebook-level note." in notebook_overview["text"]
assert "More file-level context." in notebook_overview["text"]
assert "## Math" in notebook_overview["text"]
assert "This chapter demonstrates code-first groups." in notebook_overview["text"]
assert "### Detail" not in notebook_overview["text"]
assert "detail_value = add(2, 4)" not in notebook_overview["text"]
assert chapter["resolved_kind"] == "chapter"
assert "assert add(1, 2) == 4" in chapter["text"]
assert "detail_value = add(2, 4)" in chapter["text"]
assert chapter_overview["resolved_kind"] == "chapter"
assert "This chapter demonstrates code-first groups." in chapter_overview["text"]
assert "class Calculator:" not in chapter_overview["text"]
assert "### Detail" not in chapter_overview["text"]
assert cell["resolved_kind"] == "cell"
assert "Add the doubled first value to the second." in cell["text"]
assert cell_overview["selection"]["kind"] == "implementation_context"
assert "def add(a, b):" in cell_overview["text"]
assert project["resolved_kind"] == "project"
assert any(item.endswith("01_read_single_context.ipynb") for item in project["selection"]["notebooks"])
assert file_search["resolved_kind"] == "search"
assert "search.txt:1" in file_search["text"]
assert cell_search["resolved_kind"] == "search"
assert "Nested rationale mentioning add" in cell_search["text"]

In [ ]:
#| hide
with write_demo_notebook("01_read_python_context.py") as path:
    path.write_text(
        "PUBLIC_VALUE = 3\n\n"
        "def helper(value):\n"
        "    \"\"\"Prepare a value.\n\n    Extra detail stays out of summaries.\"\"\"\n"
        "    return value + PUBLIC_VALUE\n\n"
        "def add(a, b):\n"
        "    \"\"\"Add two values.\n\n    Extra detail stays out of summaries.\"\"\"\n"
        "    return helper(a) + b\n\n"
        "def uses_add(value):\n"
        "    return add(value, 1)\n\n"
        "class Runner:\n"
        "    \"\"\"Run work.\"\"\"\n"
        "    def run(self, value):\n"
        "        return add(value, 2)\n\n"
        "def _private():\n"
        "    return None\n",
        encoding="utf-8",
    )
    py_file = python_file_context(str(path), verbose=False)
    py_symbol = python_symbol_context(str(path), "add", verbose=False)
    py_file_via_file_context = file_context(str(path), verbose=False)
    direct_file = _captured_context(str(path))
    scoped_symbol = _captured_context("add", scope=str(path))
    qualified_symbol = _captured_context(f"{path}#add")

assert py_file["kind"] == "python_file_context"
assert [item["symbol"] for item in py_file["public_symbols"]] == ["PUBLIC_VALUE", "helper", "add", "uses_add", "Runner", "Runner.run"]
assert "Prepare a value." in py_file["text"]
assert "Extra detail stays out" not in py_file["text"]
assert "_private" not in py_file["text"]
assert py_symbol["kind"] == "python_symbol_context"
assert py_symbol["symbol"] == "add"
assert "def add(a, b):" in py_symbol["source"]
assert [item["symbol"] for item in py_symbol["callers"]] == ["uses_add", "Runner.run"]
assert "helper" in py_symbol["callees"]
assert py_file_via_file_context["kind"] == "python_file_context"
assert direct_file["resolved_kind"] == "python_file"
assert scoped_symbol["resolved_kind"] == "python_symbol"
assert qualified_symbol["resolved_kind"] == "python_symbol"
assert "Callers" in scoped_symbol["text"]
assert "Callees" in qualified_symbol["text"]

In [ ]:
#| hide
with write_demo_notebook("01_read_context_ergo.ipynb") as path:
    nb = _write_read_sample_notebook(path)
    direct_cell = _captured_context(f"{path}#{nb.cells[8].id}")
    direct_index = _captured_context(f"{path}#8")
    direct_symbol = _captured_context(f"{path}#add")
    batch = _captured_context(targets=["add", "Calculator"], scope=str(path))
    failed_batch = _captured_context(targets=["add", f"{path}#does_not_exist"], scope=str(path))
    edit = _captured_context("add", scope=str(path), mode="edit", around=1)

assert direct_cell["resolved_kind"] == "cell"
assert direct_cell["target"].endswith(f"#{nb.cells[8].id}")
assert direct_index["resolved_kind"] == "cell"
assert direct_index["selection"]["location"]["cell_id"] == nb.cells[8].id
assert direct_symbol["resolved_kind"] == "symbol"
assert direct_symbol["selection"]["symbol"] == "add"
assert batch["kind"] == "context_batch"
assert [item["target"] for item in batch["results"]] == ["add", "Calculator"]
assert all(item["ok"] for item in batch["results"])
assert failed_batch["ok"] is False
assert failed_batch["results"][0]["ok"] is True
assert failed_batch["results"][1]["ok"] is False
assert failed_batch["results"][1]["candidates"]
assert edit["resolved_kind"] == "symbol"
assert edit["selection"]["kind"] == "edit_context"
assert edit["selection"]["expected_hash"]
assert len(edit["selection"]["neighbors"]) == 2

with write_demo_notebook("01_read_context_ambig_a.ipynb") as path_a:
    with write_demo_notebook("01_read_context_ambig_b.ipynb") as path_b:
        write_nb(new_nb([mk_cell("#| export\ndef shared_context_symbol():\n    return 1")]), path_a)
        write_nb(new_nb([mk_cell("#| export\ndef shared_context_symbol():\n    return 2")]), path_b)
        try:
            _captured_context("shared_context_symbol", scope=str(path_a.parent))
        except ValueError as exc:
            message, candidates = str(exc), getattr(exc, "candidates", [])
        else:
            raise AssertionError("ambiguous symbols should require a qualified target")

assert "qualified target" in message
assert f"{path_a}#shared_context_symbol" in candidates
assert f"{path_b}#shared_context_symbol" in candidates

In [ ]:
#| hide
with write_demo_notebook(
    "01_read_context_contract.ipynb",
    cells=[mk_cell("#| export\ndef add(a, b):\n    return a + b")],
) as path:
    edit = context(f"{path}#add", scope=str(path), mode="edit")
    cell = read_nb(path).cells[0]
    test_eq(edit["selection"]["expected_hash"], source_hash(cell_source(cell)))
    for mode in ("auto", "overview", "edit", "review"):
        assert context(str(path), scope=str(path), mode=mode)["mode"] == mode
    overview = context(str(path), scope=str(path), mode="overview")
    assert "definitions" not in overview["selection"]
    assert overview["selection"]["definition_count"] == 1

try:
    context("project", mode="invalid")
except ValueError as exc:
    assert "auto" in str(exc) and "review" in str(exc)
else:
    raise AssertionError("invalid mode should fail")

In [ ]:
#| hide
with write_demo_notebook("01_read_assignment_context.ipynb") as path:
    nb = new_nb([
        mk_cell("# Demo", cell_type="markdown"),
        mk_cell("#| export\nTOOL_NAMES = ','.join(['context', 'filter_context'])", cell_type="code"),
    ])
    write_nb(nb, path)
    assignment = _captured_context("TOOL_NAMES", scope=str(path))
    assert assignment["resolved_kind"] == "symbol"
    assert "TOOL_NAMES =" in assignment["text"]
    direct = symbol_context(str(path), "TOOL_NAMES", depth=0, verbose=False)
    assert direct["source"].startswith("TOOL_NAMES =")

In [ ]:
#| hide
with write_demo_notebook("01_read_filter_context.ipynb") as path:
    nb = _write_read_sample_notebook(path)
    result = filter_context(str(path), include_re="detail_value|Calculator", verbose=False)
    assert result["kind"] == "filter_context"
    assert result["total_matches"] == 5
    assert "detail_value = add(2, 4)" in result["text"]
    assert "class Calculator:" in result["text"]
    code_only = filter_context(str(path), query="type=code regex=detail_value", verbose=False)
    assert code_only["total_matches"] == 1
    assert code_only["matches"][0]["cell_type"] == "code"

    summary = filter_context(str(path), view="summary", verbose=False)
    assert summary["total_matches"] == len(nb.cells)
    assert summary["matches"][0]["summary"].startswith(f"{path}#")
    assert "idx=0 type=markdown" in summary["text"]

    numbered = filter_context(str(path), query=f"id={nb.cells[8].id}", view="cell", verbose=False)
    assert numbered["total_matches"] == 1
    assert "1 | def add(a, b):" in numbered["text"]
    assert "2 |" in numbered["text"]

    errors = filter_context(str(path), query="errors", verbose=False)
    assert errors["total_matches"] == 1
    assert errors["matches"][0]["error"] == "RuntimeError: boom"

    exports = filter_context(str(path), query="export", view="summary", verbose=False)
    assert exports["total_matches"] == 3
    assert all(item["semantic_type"] == "exported code" for item in exports["matches"])

    outline = filter_context(str(path), query="headers", view="summary", verbose=False)
    assert [item["cell_idx"] for item in outline["matches"]] == [0, 3, 16]
    assert "### Detail" in outline["text"]

    section = filter_context(str(path), query="chapter=Detail", verbose=False)
    assert section["total_matches"] == 3
    assert "Nested rationale" in section["text"]
    assert "detail_value = add(2, 4)" in section["text"]
    assert "RuntimeError" in section["text"]

    grep = filter_context(str(path), query="regex=detail_value", view="summary", before=1, after=1, verbose=False)
    assert grep["total_matches"] == 1
    assert grep["matches"][0]["before"][0]["cell_idx"] == 16
    assert grep["matches"][0]["after"][0]["cell_idx"] == 18

In [ ]:
#| hide
with write_demo_notebook("01_read_doc.ipynb") as path:
    nb = new_nb([
        mk_cell("# Demo project\nFile-level note.", cell_type="markdown"),
        mk_cell("#| export\ndef add(a, b):\n    \"\"\"Add two values.\"\"\"\n    return a + b", cell_type="code"),
        mk_cell("## Addition\nThis explains add.", cell_type="markdown"),
        mk_cell("add(2, 3)", cell_type="code"),
        mk_cell("assert add(1, 2) == 3", cell_type="code"),
    ])
    write_nb(nb, path)
    out = StringIO()
    with redirect_stdout(out):
        result = symbol_context(str(path), "add", depth=0)
    text = out.getvalue()
    assert result["kind"] == "symbol_context"
    assert "Symbol context: add" in text
    assert "def add(a, b):" in text
    assert "This explains add." in text
    assert "Docs" in text
    assert "add(2, 3)" in text
    assert "assert add(1, 2) == 3" in text
    assert text.index("def add(a, b):") < text.index("This explains add.") < text.index("Examples/tests")
    assert "Callers" not in text
    focused = _captured_context("add", scope=str(path))
    assert "Implementation" in focused["text"]
    assert "Examples/tests" in focused["text"]
    assert "def add(a, b):" in focused["text"]
    focused_overview = _captured_context("add", scope=str(path), mode="overview")
    assert "def add(a, b):" in focused_overview["text"]

In [ ]:
#| hide
with write_demo_notebook("01_read_edit_context.ipynb") as path:
    nb = new_nb([
        mk_cell("#| export\ndef add(a, b):\n    return a + b"),
        mk_cell("This explains add.", cell_type="markdown"),
        mk_cell("add(2, 3)"),
        mk_cell("assert add(1, 2) == 3"),
    ])
    write_nb(nb, path)
    edit = _captured_context("add", scope=str(path), mode="edit")

assert edit["selection"]["docs"][0]["source"].endswith("This explains add.")
assert {item["source"] for item in edit["selection"]["examples"]} == {"add(2, 3)", "assert add(1, 2) == 3"}
assert "Examples/tests" in edit["selection"]["text"]

In [ ]:
#| hide
with write_demo_notebook("01_read_sample.ipynb") as path:
    _write_read_sample_notebook(path)
    out = StringIO()
    with redirect_stdout(out):
        result = file_context(str(path))
    text = out.getvalue()
    assert result["kind"] == "file_context"
    assert "# Sample tool" in text
    assert "Notebook-level note." in text
    assert "import math" in text
    assert "def add(a, b):" in text
    assert "Add values." in text
    assert "class Calculator:" in text
    assert "    def total(self, value):" in text
    assert "Add value to the base." in text

    filtered = file_context(str(path), include_re="Calculator", verbose=False)
    assert "Calculator" in filtered["text"]
    assert "def add(a, b):" not in filtered["text"]

    excluded = file_context(str(path), exclude_re="Calculator", verbose=False)
    assert "def add(a, b):" in excluded["text"]
    assert "class Calculator:" not in excluded["text"]

In [ ]:
#| hide
with write_demo_notebook("01_read_sample.ipynb") as path:
    _write_read_sample_notebook(path)
    out = StringIO()
    with redirect_stdout(out):
        result = chapter_context(str(path), name="Math")
    text = out.getvalue()
    assert result["kind"] == "chapter_context"
    assert "This chapter demonstrates code-first groups." in text
    assert "assert add(1, 2) == 4" in text
    assert "detail_value = add(2, 4)" in text
    assert "1 |" not in text

    overview = chapter_context(str(path), name="Math", overview=True, verbose=False)
    assert "This chapter demonstrates code-first groups." in overview["text"]
    assert "This chapter demonstrates code-first groups." in overview["text"]
    assert "class Calculator:" not in overview["text"]
    assert "### Detail" not in overview["text"]

    filtered = chapter_context(str(path), name="Math", cell_type="code", include_re="detail_value|Calculator", verbose=False)
    assert "class Calculator:" in filtered["text"]
    assert "detail_value = add(2, 4)" in filtered["text"]
    assert "This chapter demonstrates code-first groups." not in filtered["text"]
    assert all(item["cell_type"] == "code" for item in filtered["cells"])

    out = StringIO()
    with redirect_stdout(out):
        chapter_context(str(path), name="Detail")
    text = out.getvalue()
    assert "Nested rationale" in text
    assert "detail_value = add(2, 4)" in text

In [ ]:
#| hide
with write_demo_notebook("01_read_sample.ipynb") as path:
    _write_read_sample_notebook(path)
    project = project_context(str(path), verbose=False)
    assert project["kind"] == "project_context"
    assert any(item.endswith("01_read_sample.ipynb") for item in project["notebooks"])
    assert "Sample tool" in project["text"]

    shallow = symbol_context(str(path), "add", depth=0, verbose=False)
    assert "def add(a, b):" in shallow["text"]
    assert "Callers" not in shallow["text"]
    assert "Callees depth" not in shallow["text"]

    deep = symbol_context(str(path), "add", depth=1, verbose=False)
    assert "double" in deep["text"]
    assert "Output:" in deep["text"]
    assert "5" in deep["text"]

### Symbol documentation

`symbol_context` answers the symbol-first question: "what should I know before changing this implementation?" It finds the defining function, class, or method, then reads the local group in source order: implementation, trailing Docs, example/test cells, callers, and optional callee summaries. The group ends at the next exported code cell, which keeps neighboring symbol stories separate.